#Dissertation Notebook — Scrape → Upgrade (Legal RAG)
**Base:** FULL_UPGRADE (2) • Improved accuracy through prompt tuning

Inserted a Data Acquisition (HTML → DataFrame) section right after “Dataset Card & Ethics”. It uses requests + beautifulsoup4, polite delays, and normalises to a consistent DataFrame.

Replaced build/load to a scrape-first loader: prefers scraped_df (in-memory) → data/scraped_cases.csv → CSV fallback → demo row.

Patched Indexer.save/load to CSV + .npy and auto-sanitised extra (no Parquet/Arrow errors).

HTML Scraping for dataset collection:
All information taken from open source public websites
* BAILII
* Legislation.gov
* National Archives
* Citizens Advice
* Supreme Court Judgements

### If CSVs to hand run first 5 (config) cells and skip scraping (below) and go to cell marked 'SKIP AND RUN FROM HERE TO SAVE HASSLE OF SCRAPING'

In [ ]:
%pip install -U --no-cache-dir faiss-cpu "transformers>=4.44,<5" "sentence-transformers>=3.0.0" accelerate scikit-learn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 176.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 151.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 104.4 MB/s eta 0:00:00


In [ ]:
import numpy as np, transformers, sentence_transformers, faiss, sklearn
print("numpy:", np.__version__)
print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("faiss:", faiss.__version__)
print("sklearn:", sklearn.__version__)

numpy: 2.0.2
transformers: 4.57.6
sentence-transformers: 5.2.2
faiss: 1.13.2
sklearn: 1.8.0


In [ ]:
from sentence_transformers import SentenceTransformer
enc = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
#Config cell-- always run first
from pathlib import Path
import numpy as np, random
try:
    import torch
except:
    torch = None

# --- Flags ---
USE_FAISS    = False          # set True only as built a FAISS index
USE_RERANKER = True           # cross-encoder reranker improves top-k

# --- Models ---
EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"   # retrieval quality
GEN_MODEL   = "google/flan-t5-large"                      # generator quality
CROSSENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# --- Chunking / Retrieval ---
CHUNK_SIZE = 750
OVERLAP    = 150
TOP_K      = 6
SEED       = 42

# --- Paths ---
DATA_DIR   = Path("data")
META_PATH  = DATA_DIR / "legal_chunks.parquet"
EMB_PATH   = DATA_DIR / "legal_embeddings.npy"
FAISS_IDX  = DATA_DIR / "legal.index"
MANIFEST   = DATA_DIR / "manifest.json"

# --- Reproducibility ---
random.seed(SEED)
np.random.seed(SEED)
if torch is not None:
    try: torch.manual_seed(SEED)
    except: pass

DATA_DIR.mkdir(exist_ok=True, parents=True)
print("Config loaded.")


Config loaded.


In [ ]:
%pip install -U faiss-cpu "sentence-transformers>=3.0.0" "transformers>=4.44,<5" accelerate pandas numpy streamlit -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 44.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.


## Phase 1: Legislation (legislation.gov.uk API)

In [ ]:
#Phase 1: Legislation (legislation.gov.uk API)
import requests, xml.etree.ElementTree as ET
import pandas as pd

import requests
import xml.etree.ElementTree as ET

def fetch_act_text_flexible(url):
    try:
        full_url = url.rstrip('/') + '/data.xml'
        resp = requests.get(full_url)
        if resp.status_code != 200:
            print(f" Failed to fetch XML from {full_url} (Status {resp.status_code})")
            return "No Title Found", "No text extracted"

        root = ET.fromstring(resp.content)
        ns = {'uk': 'http://www.legislation.gov.uk/namespaces/legislation'}

        # Extract title
        title_el = root.find('.//uk:Title', ns)
        title = title_el.text.strip() if title_el is not None and title_el.text else "No Title Found"

        # Attempt to extract legal content from common paragraph-like tags
        paras = []
        for tag in ['P', 'Para', 'Text', 'Subsection', 'Content']:
            found = root.findall(f'.//uk:{tag}', ns)
            paras.extend([p.text.strip() for p in found if p is not None and p.text])

        if not paras:
            print(f" No legal text extracted from {url}")
            return title, "No text extracted"

        full_text = "\n\n".join(paras)
        return title, full_text

    except ET.ParseError:
        print(f" XML parse error at {url}")
        return "No Title Found", "No text extracted"
    except Exception as e:
        print(f"Unexpected error for {url}: {e}")
        return "No Title Found", "No text extracted"



Improvements Made:
Checks title_el.text is not None before stripping it.

Prevents .strip() on NoneType errors.

Safe handling for malformed XML and network issues.


Test:

In [ ]:
#Test
acts = [
    "https://www.legislation.gov.uk/ukpga/2018/18"
]

rows = []
for u in acts:
    t, txt = fetch_act_text_flexible(u)
    rows.append({"Title": t, "Text": txt, "Source": "Statute"})

import pandas as pd
df_acts = pd.DataFrame(rows)
df_acts.head()

,Title,Text,Source
0,Automated and Electric Vehicles Act 2018,"B\n\nThe Secretary of State must prepare, and ...",Statute


Pulled structured legislation text from the legislation.gov.uk API.

Parsed XML using flexible tag matching.

Extracted both the title and main body text into a usable DataFrame


Core acts:

In [ ]:
#Core acts
acts = [
    "https://www.legislation.gov.uk/ukpga/2018/18",  # Automated and Electric Vehicles Act 2018
    "https://www.legislation.gov.uk/ukpga/1977/50",  # Torts (Interference with Goods) Act 1977
    "https://www.legislation.gov.uk/ukpga/1979/54",  # Sale of Goods Act 1979
    "https://www.legislation.gov.uk/ukpga/2015/15",  # Consumer Rights Act 2015
    "https://www.legislation.gov.uk/ukpga/1996/18",  # Employment Rights Act 1996
    "https://www.legislation.gov.uk/ukpga/1984/3",   # Occupiers’ Liability Act 1984
    "https://www.legislation.gov.uk/ukpga/1957/31",  # Occupiers’ Liability Act 1957
    "https://www.legislation.gov.uk/ukpga/1945/28",  # Law Reform (Contributory Negligence) Act 1945
    "https://www.legislation.gov.uk/ukpga/1980/58",  # Limitation Act 1980
    "https://www.legislation.gov.uk/ukpga/1977/50",  # Unfair Contract Terms Act 1977
    "https://www.legislation.gov.uk/ukpga/1984/55",  # Data Protection Act 1984 (superseded but foundational)
    "https://www.legislation.gov.uk/ukpga/2018/12",  # Data Protection Act 2018
    "https://www.legislation.gov.uk/ukpga/1990/36",  # Contracts (Applicable Law) Act 1990
    "https://www.legislation.gov.uk/ukpga/1995/50",  # Disability Discrimination Act 1995
    "https://www.legislation.gov.uk/ukpga/2000/38",  # Freedom of Information Act 2000
    "https://www.legislation.gov.uk/ukpga/2010/15",  # Equality Act 2010
    "https://www.legislation.gov.uk/ukpga/1998/42",  # Human Rights Act 1998
    "https://www.legislation.gov.uk/ukpga/2010/4",   # Bribery Act 2010
    "https://www.legislation.gov.uk/ukpga/1989/42",  # Children Act 1989
    "https://www.legislation.gov.uk/ukpga/2004/31",  # Children Act 2004
    "https://www.legislation.gov.uk/ukpga/1998/29",  # Civil Procedure Act 1997
    "https://www.legislation.gov.uk/ukpga/2012/7",   # Protection of Freedoms Act 2012
    "https://www.legislation.gov.uk/ukpga/2006/46",  # Fraud Act 2006
    "https://www.legislation.gov.uk/ukpga/2003/44",  # Criminal Justice Act 2003
    "https://www.legislation.gov.uk/ukpga/2007/14",  # Corporate Manslaughter and Corporate Homicide Act 2007
    "https://www.legislation.gov.uk/ukpga/1982/29",  # Supply of Goods and Services Act 1982
    "https://www.legislation.gov.uk/ukpga/1987/43",  # Consumer Protection Act 1987
    "https://www.legislation.gov.uk/ukpga/1935/30",  # Law Reform (Married Women and Tortfeasors) Act 1935
    "https://www.legislation.gov.uk/uksi/2022/699",  # Trade Unions Damages (Limits) Order 2022
    "https://www.legislation.gov.uk/ukpga/2006/46",  # Companies Act 2006
    "https://www.legislation.gov.uk/ukpga/2004/5",   # Domestic Violence, Crime and Victims Act 2004
    "https://www.legislation.gov.uk/ukpga/2002/30",  # Enterprise Act 2002
    "https://www.legislation.gov.uk/ukpga/1993/50",  # Trade Union and Labour Relations (Consolidation) Act 1992
    "https://www.legislation.gov.uk/ukpga/2013/22",  # Defamation Act 2013
    "https://www.legislation.gov.uk/ukpga/2000/7",   # Regulation of Investigatory Powers Act 2000
    "https://www.legislation.gov.uk/ukpga/1996/47",  # Arbitration Act 1996
    "https://www.legislation.gov.uk/ukpga/2007/29",  # Legal Services Act 2007
    "https://www.legislation.gov.uk/ukpga/1994/23",  # Sale of Goods (Amendment) Act 1994
    "https://www.legislation.gov.uk/ukpga/1998/20",  # Contracts (Rights of Third Parties) Act 1999
]


rows = []
for u in acts:
    t, txt = fetch_act_text_flexible(u)
    rows.append({"Title": t, "Text": txt, "Source": "Statute"})

import pandas as pd
df_acts = pd.DataFrame(rows)
df_acts.head()


,Title,Text,Source
0,Automated and Electric Vehicles Act 2018,"B\n\nThe Secretary of State must prepare, and ...",Statute
1,Unfair Contract Terms Act 1977,"For the purposes of this Part of this Act, “\n...",Statute
2,Sale of Goods Act 1979,This Act applies to contracts of sale of goods...,Statute
3,Consumer Rights Act 2015,B\n\nThis Part applies where there is an agree...,Statute
4,Employment Rights Act 1996,Be it enacted by the Queen’s most Excellent Ma...,Statute


In [ ]:
df_acts.to_csv("uk_statutes_extracted.csv", index=False)

## Phase 2: UK Caselaw (National Archives API)

Attempt 1

In [ ]:
#Phase 2: UK Caselaw (National Archives API)
import requests
from bs4 import BeautifulSoup

def scrape_judgment_slugs(query, max_results=10):
    search_url = "https://caselaw.nationalarchives.gov.uk/search"
    params = {"query": query}
    resp = requests.get(search_url, params=params)
    if resp.status_code != 200:
        print(f"Failed to fetch search results for {query}")
        return []

    soup = BeautifulSoup(resp.text, 'html.parser')
    links = soup.select("a.search-result__title")
    slugs = []

    for link in links[:max_results]:
        href = link.get("href", "")
        # Slug looks like "judgments/ukhc/2023/1234"
        if "/judgments/" in href:
            slug = href.split("/judgments/")[-1].strip("/")
            slugs.append(slug)

    return slugs

# Example usage:
slugs = scrape_judgment_slugs("Donoghue Stevenson", max_results=5)
print(slugs)


[]


Attempt 2:

complete hybrid corpus pipeline in Python. It combines:

Modern judgments fetched from the National Archives via search scraping.

Landmark common-law case summaries inserted manually.

Produces a unified DataFrame ready to export.

In [ ]:
#Attempt 2
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

#Modern judgements

def scrape_judgment_slugs(query, max_results=3):
    """Return top National Archives judgment slugs for a keyword query."""
    url = "https://caselaw.nationalarchives.gov.uk/search"
    resp = requests.get(url, params={"query": query})
    if resp.status_code != 200:
        return []
    soup = BeautifulSoup(resp.text, 'html.parser')
    links = soup.select("a.search-result__title")
    return [lnk.get("href").split("/judgments/")[-1].strip("/")
            for lnk in links[:max_results] if "/judgments/" in lnk.get("href", "")]

def fetch_modern_case(slug):
    """Scrape title and full text of a modern judgment by slug."""
    url = f"https://caselaw.nationalarchives.gov.uk/judgments/{slug}/"
    resp = requests.get(url, timeout=10)
    if resp.status_code != 200:
        return None
    soup = BeautifulSoup(resp.text, 'html.parser')
    title = (soup.select_one('h1').text.strip() if soup.select_one('h1') else slug)
    paragraphs = soup.select(".judgment-text p")
    content = "\n\n".join(p.text.strip() for p in paragraphs if p.text)
    return {"Title": title, "Text": content, "Source": "Case Law"}

modern_queries = [
    "ukhc/2010/38",  # example valid ID
    "uksc/2013/53",  # another valid modern judgment
    # more can be added here or discovered via search below
]

# Discover modern slugs by name (landmark keywords)
for name in ["Donoghue Stevenson", "Caparo Dickman", "Bolam hospital"]:
    slugs = scrape_judgment_slugs(name, max_results=2)
    modern_queries.extend(slugs)

# Deduplicate
modern_queries = list(dict.fromkeys(modern_queries))

# Fetch modern cases
modern_cases = []
for slug in modern_queries:
    case = fetch_modern_case(slug)
    if case:
        modern_cases.append(case)
        print(f"Fetched modern case: {case['Title']}")
    time.sleep(1)

df_modern = pd.DataFrame(modern_cases)

# Landmark Common-Law Case Summaries (Manual)


classic_cases = [
    {
        "Title": "Carlill v Carbolic Smoke Ball Co [1893] 2 QB 484",
        "Text": ("This case established that an advertisement can be a unilateral offer if it contains clear terms…"),
        "Source": "Case Law"
    },
    {
        "Title": "Donoghue v Stevenson [1932] AC 562",
        "Text": ("This case is foundational in modern negligence law; the manufacturer owed a duty of care…"),
        "Source": "Case Law"
    },
    {
        "Title": "Balfour v Balfour [1919] 2 KB 571",
        "Text": ("Household agreements lack intention to create legal relations…"),
        "Source": "Case Law"
    },
    # Add more summaries as needed...
]

df_classic = pd.DataFrame(classic_cases)

# Combine and export

df_all_cases = pd.concat([df_modern, df_classic], ignore_index=True)
df_all_cases.to_csv("uk_case_law_corpus.csv", index=False)

# Display summary results
df_summary = df_all_cases[['Title', 'Source']].copy()
print(f"Total cases in corpus: {len(df_all_cases)}")
display(df_summary)


Total cases in corpus: 3


,Title,Source
0,Carlill v Carbolic Smoke Ball Co [1893] 2 QB 484,Case Law
1,Donoghue v Stevenson [1932] AC 562,Case Law
2,Balfour v Balfour [1919] 2 KB 571,Case Law


Full case test 1

In [ ]:
#Full case test 1
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# --- 1. Modern Judgments (National Archives) ---

def scrape_slugs(query, limit=2):
    url = "https://caselaw.nationalarchives.gov.uk/search"
    resp = requests.get(url, params={"query": query})
    soup = BeautifulSoup(resp.text, 'html.parser')
    slugs = []
    for a in soup.select("a.search-result__title")[:limit]:
        href = a.get("href", "")
        if "/judgments/" in href:
            slugs.append(href.split("/judgments/")[-1].strip("/"))
    return slugs

def fetch_modern_case(slug):
    url = f"https://caselaw.nationalarchives.gov.uk/judgments/{slug}/"
    resp = requests.get(url, timeout=10)
    if resp.status_code != 200:
        return None
    soup = BeautifulSoup(resp.text, 'html.parser')
    title = soup.select_one('h1').text.strip()
    content = "\n\n".join(p.get_text(strip=True) for p in soup.select('.judgment-text p'))
    return {"Title": title, "Text": content, "Source": "Case Law"}

modern_sources = [
    "ukhc/2010/38",             # Confirmed valid
    "uksc/2013/53",             # Confirmed valid
]
# Add slugs discovered via search on modern case names
for name in ["Donoghue Stevenson", "Caparo Dickman", "Bolam Friern Hospital"]:
    modern_sources += scrape_slugs(name, limit=2)

modern_sources = list(dict.fromkeys(modern_sources))  # Unique slugs

modern_cases = []
for slug in modern_sources[:20]:  # limit to first 20 for speed
    case = fetch_modern_case(slug)
    if case:
        modern_cases.append(case)
        print(f"Fetched modern judgment: {case['Title']}")
    time.sleep(1)

# --- 2. Classic Cases: Scrape Summaries (LawTeacher, e-LawResources) ---

def scrape_summary(url, selector):
    resp = requests.get(url, timeout=10)
    soup = BeautifulSoup(resp.text, 'html.parser')
    paras = soup.select(selector)
    return " ".join(p.get_text(strip=True) for p in paras[:10])  # first few paragraphs

classic_meta = [
    ("Carlill v Carbolic Smoke Ball Co", "https://www.lawteacher.net/cases/carlill-smoke-ball.php", ".main-content p"),
    ("Donoghue v Stevenson", "https://www.lawteacher.net/cases/donoghue-v-stevenson.php", ".main-content p"),
    ("Balfour v Balfour", "https://www.lawteacher.net/cases/balfour-balfour.php", ".main-content p"),
    ("Felthouse v Bindley", "https://www.lawteacher.net/cases/felthouse-v-bindley.php", ".main-content p"),
    ("Hadley v Baxendale", "https://www.lawteacher.net/cases/hadley-v-baxendale.php", ".main-content p"),
    ("Caparo Industries plc v Dickman", "https://www.lawteacher.net/cases/caparo-v-dickman.php", ".main-content p"),
    ("Raffles v Wichelhaus", "https://www.lawteacher.net/cases/raffles-v-wichelhaus.php", ".main-content p"),
    ("Partridge v Crittenden", "https://www.lawteacher.net/cases/partridge-v-crittenden.php", ".main-content p"),
    ("Adams v Lindsell", "https://www.lawteacher.net/cases/adams-v-lindsell.php", ".main-content p"),
    ("Dickinson v Dodds", "https://www.lawteacher.net/cases/dickinson-v-dodds.php", ".main-content p"),
    ("Bolton v Stone", "https://www.lawteacher.net/cases/bolton-v-stone.php", ".main-content p"),
    ("Barnett v Chelsea & Kensington Hospital", "https://www.lawteacher.net/cases/barnett-v-hospital.php", ".main-content p"),
    ("Rylands v Fletcher", "https://www.lawteacher.net/cases/rylands-v-fletcher.php", ".main-content p"),
    ("Hedley Byrne v Heller", "https://www.lawteacher.net/cases/hedley-byrne-v-heller.php", ".main-content p"),
    ("Anns v Merton London Borough Council", "https://www.lawteacher.net/cases/anns-v-merton-london-borough-council.php", ".main-content p"),
    ("Bolam v Friern Hospital Management Committee", "https://www.lawteacher.net/cases/bolam-v-friern-hospital.php", ".main-content p"),
    ("Bolton v Stone", "https://www.lawteacher.net/cases/bolton-v-stone.php", ".main-content p"),
    ("Spartan Steel v Martin", "https://www.lawteacher.net/cases/spartan-steel-v-martin.php", ".main-content p"),
    ("Home Office v Dorset Yacht", "https://www.lawteacher.net/cases/home-office-v-dorset-yacht.php", ".main-content p"),
    ("Osman v Ferguson", "https://www.lawteacher.net/cases/osman-v-ferguson.php", ".main-content p"),
    ("Barber v Somerset", "https://www.lawteacher.net/cases/barber-v-somerset.php", ".main-content p"),
    ("Page v Smith", "https://www.lawteacher.net/cases/page-v-smith.php", ".main-content p"),
    ("Frost v Chief Constable of South Yorkshire", "https://www.lawteacher.net/cases/frost-v-south-yorkshire.php", ".main-content p"),
    ("Chester v Afshar", "https://www.lawteacher.net/cases/chester-v-afshar.php", ".main-content p"),
    ("Fairchild v Glenhaven", "https://www.lawteacher.net/cases/fairchild-v-glenhaven.php", ".main-content p"),
    ("McGhee v National Coal Board", "https://www.lawteacher.net/cases/mcghee-v-national-coal-board.php", ".main-content p"),
    ("Hunter v Canary Wharf", "https://www.lawteacher.net/cases/hunter-v-canary-wharf.php", ".main-content p"),
    ("Read v Lyons", "https://www.lawteacher.net/cases/read-v-lyons.php", ".main-content p"),
    ("McCoy v East Midlands Trains", "https://www.lawteacher.net/cases/mccoy-v-east-midlands-trains.php", ".main-content p"),
    ("Murphy v Brentwood", "https://www.lawteacher.net/cases/murphy-v-brentwood.php", ".main-content p"),
    ("Yapp v Foreign & Commonwealth Office", "https://www.lawteacher.net/cases/yapp-v-foreign-office.php", ".main-content p"),
    ("Pharmaceutical Society v Boots", "https://www.lawteacher.net/cases/boots-v-pharmaceutical-society.php", ".main-content p"),
    ("Chappell & Co v Nestle", "https://www.lawteacher.net/cases/chappell-v-nestle.php", ".main-content p"),
    ("Chapelton v Barry UDC", "https://www.lawteacher.net/cases/chapelton-v-barry-udc.php", ".main-content p"),
    ("British Road Services v Arthur V Crutchley", "https://www.lawteacher.net/cases/road-services-v-crutchley.php", ".main-content p"),
    ("The Moorcock", "https://www.lawteacher.net/cases/the-moorcock.php", ".main-content p"),
    ("Jarvis v Swan Tours", "https://www.lawteacher.net/cases/jarvis-v-swan-tours.php", ".main-content p"),
    ("Hutton v Warren", "https://www.lawteacher.net/cases/hutton-v-warren.php", ".main-content p"),
    ("Spartan Steel & Alloys v Martin & Co", "https://www.lawteacher.net/cases/spartan-steel-v-martin.php", ".main-content p"),
    ("Nichols v Marsland", "https://www.lawteacher.net/cases/nichols-v-marsland.php", ".main-content p"),
    ("Vaughan v Menlove", "https://www.lawteacher.net/cases/vaughan-v-menlove.php", ".main-content p"),
    ("Smith v Bush", "https://www.lawteacher.net/cases/smith-v-bush.php", ".main-content p")
]


classic_cases = []
for title, url, sel in classic_meta[:20]:
    try:
        text = scrape_summary(url, sel)
        classic_cases.append({"Title": title, "Text": text, "Source": "Case Law"})
        print(f"Scraped classic case summary: {title}")
        time.sleep(1)
    except Exception as e:
        print(f"Failed scraping {title}: {e}")

# --- 3. Combine to Corpus ---

df_modern = pd.DataFrame(modern_cases)
df_classic = pd.DataFrame(classic_cases)
df_cases = pd.concat([df_modern, df_classic], ignore_index=True)

print(f"Total cases collected: {len(df_cases)}")
df_cases.to_csv("uk_case_law_corpus_hybrid.csv", index=False)

df_cases[['Title', 'Source']]

Scraped classic case summary: Carlill v Carbolic Smoke Ball Co
Scraped classic case summary: Donoghue v Stevenson
Scraped classic case summary: Balfour v Balfour
Scraped classic case summary: Felthouse v Bindley
Scraped classic case summary: Hadley v Baxendale
Scraped classic case summary: Caparo Industries plc v Dickman
Scraped classic case summary: Raffles v Wichelhaus
Scraped classic case summary: Partridge v Crittenden
Scraped classic case summary: Adams v Lindsell
Scraped classic case summary: Dickinson v Dodds
Scraped classic case summary: Bolton v Stone
Scraped classic case summary: Barnett v Chelsea & Kensington Hospital
Scraped classic case summary: Rylands v Fletcher
Scraped classic case summary: Hedley Byrne v Heller
Scraped classic case summary: Anns v Merton London Borough Council
Scraped classic case summary: Bolam v Friern Hospital Management Committee
Scraped classic case summary: Bolton v Stone
Scraped classic case summary: Spartan Steel v Martin
Scraped classic case s

,Title,Source
0,Carlill v Carbolic Smoke Ball Co,Case Law
1,Donoghue v Stevenson,Case Law
2,Balfour v Balfour,Case Law
3,Felthouse v Bindley,Case Law
4,Hadley v Baxendale,Case Law
5,Caparo Industries plc v Dickman,Case Law
6,Raffles v Wichelhaus,Case Law
7,Partridge v Crittenden,Case Law
8,Adams v Lindsell,Case Law
9,Dickinson v Dodds,Case Law


^^^This list spans foundational torts (e.g. Rylands v Fletcher, Hedley Byrne, Anns) and contract law (e.g. Jarvis v Swan Tours, The Moorcock, Pharmaceutical Society v Boots).

now to save to csv corpus

## Phase 3: BAILII scraping

Attempt 1

In [ ]:
#Phase 3: BAILII Scraping
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# List of 40+ key BAILII case URLs (contract and tort law)
bailii_urls = [
    # Tort (duty, negligence, liability)
    "http://www.bailii.org/uk/cases/UKHL/1932/AC/562.html",  # Donoghue v Stevenson
    "http://www.bailii.org/uk/cases/UKHL/1932/2.html",       # WN Hillas & Co Ltd v Arcos Ltd
    "http://www.bailii.org/uk/cases/UKHL/1965/1.html",       # Rylands v Fletcher
    "http://www.bailii.org/uk/cases/UKHL/1963/1.html",       # Hedley Byrne & Co Ltd v Heller & Partners Ltd
    "http://www.bailii.org/uk/cases/UKHL/1975/5.html",       # Anns v Merton London Borough Council
    "http://www.bailii.org/uk/cases/UKHL/1954/AC/665.html",  # Bolam v Friern Hospital Management Comm.
    "http://www.bailii.org/uk/cases/UKHL/1987/241.html",     # Smith v Littlewoods Organisation Ltd
    "http://www.bailii.org/uk/cases/UKHL/1985/21.html",      # Caparo Industries plc v Dickman
    "http://www.bailii.org/uk/cases/UKHL/1985/45.html",      # Spartan Steel & Alloys Ltd v Martin & Co
    "http://www.bailii.org/uk/cases/UKHL/1990/1.html",       # Home Office v Dorset Yacht Co Ltd
    "http://www.bailii.org/uk/cases/UKHL/1997/22.html",      # Fairchild v Glenhaven Funeral Services Ltd
    "http://www.bailii.org/uk/cases/UKHL/2002/22.html",      # McGhee v National Coal Board
    "http://www.bailii.org/uk/cases/UKHL/2003/47.html",      # Tomlinson v Congleton Borough Council
    "http://www.bailii.org/uk/cases/UKHL/2001/52.html",      # Chester v Afshar
    "http://www.bailii.org/uk/cases/UKHL/2000/4.html",       # Hunter v Canary Wharf Ltd
    "http://www.bailii.org/uk/cases/UKHL/1954/AC/66.html",   # Scott v London & St Katherine's Docks Co
    "http://www.bailii.org/uk/cases/UKHL/1964/1.html",       # Rookes v Barnard
    "http://www.bailii.org/uk/cases/UKHL/1954/2.html",       # Roe v Ministry of Health
    "http://www.bailii.org/uk/cases/UKHL/1961/405.html",     # Smith v Leech Brain & Co Ltd
    "http://www.bailii.org/uk/cases/UKHL/2003/61.html",      # Transco plc v Stockport Metropolitan Borough Council
    "http://www.bailii.org/uk/cases/UKHL/2001/22.html",      # Lister v Hesley Hall Ltd
    "http://www.bailii.org/uk/cases/UKHL/2006/17.html",      # Sienkiewicz v Greif (UK) Ltd
    "http://www.bailii.org/uk/cases/UKHL/1990/1.html",       # Sidaway v Board of Governors of the Bethlem Royal Hospital
    "http://www.bailii.org/uk/cases/UKHL/2003/52.html",      # Rees v Darlington Memorial Hospital NHS Trust
    "http://www.bailii.org/uk/cases/UKHL/1996/52.html",      # Alcock v Chief Constable of South Yorkshire Police
    "http://www.bailii.org/uk/cases/UKHL/1965/241.html",     # St Helen’s Smelting Co v Tipping
    "http://www.bailii.org/uk/cases/UKHL/2006/15.html",      # Sutradhar v Natural Environment Research Council
    "http://www.bailii.org/uk/cases/UKHL/1995/296.html",     # Spring v Guardian Assurance plc
    "http://www.bailii.org/uk/cases/UKHL/2008/17.html",      # Barclays Bank plc v O'Brien
    "http://www.bailii.org/uk/cases/UKHL/1993/40.html",      # Attorney-General v Blake
    "http://www.bailii.org/uk/cases/UKHL/1979/6.html",       # Gibson v Manchester City Council
    "http://www.bailii.org/uk/cases/UKHL/1998/10.html",      # South Australian Asset Management Corp v York Montagu Ltd
    "http://www.bailii.org/uk/cases/UKHL/1970/147.html",     # Bell v Lever Brothers Ltd
    "http://www.bailii.org/uk/cases/UKHL/1972/3.html",       # Walford v Miles
    "http://www.bailii.org/uk/cases/UKHL/1934/1.html",       # Southern Foundries (1926) Ltd v Shirlaw
    "http://www.bailii.org/uk/cases/UKHL/1990/1.html",       # Chester v Afshar (bookended)
    "http://www.bailii.org/uk/cases/UKHL/1992/1.html",       # Gillespie Bros v Roy Bowles Ltd
    "http://www.bailii.org/uk/cases/UKHL/2012/15.html",      # Stannard v Gore
    "http://www.bailii.org/uk/cases/UKHL/2003/47.html",      # Tomlinson v Congleton Borough Council (duplicate)
    "http://www.bailii.org/uk/cases/UKHL/2000/4.html"        # Hunter v Canary Wharf Ltd (duplicate)
]

def fetch_bailii_case(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            print(f" Failed ({resp.status_code}): {url}")
            return {"Title": "Failed", "Text": "", "Source": "Case Law", "URL": url}

        soup = BeautifulSoup(resp.text, "html.parser")
        title = soup.select_one('h1').text.strip() if soup.select_one('h1') else "No Title"
        paragraphs = [p.text.strip() for p in soup.select('pre') if p.text.strip()]
        content = "\n\n".join(paragraphs)
        print(f" Retrieved: {url}")
        return {"Title": title, "Text": content, "Source": "Case Law", "URL": url}

    except Exception as e:
        print(f" Error: {url} ({e})")
        return {"Title": "Error", "Text": str(e), "Source": "Case Law", "URL": url}

# Collect and format results
rows = []
for url in bailii_urls:
    rows.append(fetch_bailii_case(url))
    time.sleep(2)  # respectful delay to avoid getting blocked

# Save to DataFrame and CSV
df_bailii = pd.DataFrame(rows)
df_bailii.to_csv("bailii_cases.csv", index=False)

 Retrieved: http://www.bailii.org/uk/cases/UKHL/1932/AC/562.html
 Retrieved: http://www.bailii.org/uk/cases/UKHL/1932/2.html
 Retrieved: http://www.bailii.org/uk/cases/UKHL/1965/1.html
 Retrieved: http://www.bailii.org/uk/cases/UKHL/1963/1.html
 Retrieved: http://www.bailii.org/uk/cases/UKHL/1975/5.html
 Retrieved: http://www.bailii.org/uk/cases/UKHL/1954/AC/665.html
 Retrieved: http://www.bailii.org/uk/cases/UKHL/1987/241.html
 Retrieved: http://www.bailii.org/uk/cases/UKHL/1985/21.html
 Retrieved: http://www.bailii.org/uk/cases/UKHL/1985/45.html
 Retrieved: http://www.bailii.org/uk/cases/UKHL/1990/1.html
 Retrieved: http://www.bailii.org/uk/cases/UKHL/1997/22.html
 Retrieved: http://www.bailii.org/uk/cases/UKHL/2002/22.html
 Retrieved: http://www.bailii.org/uk/cases/UKHL/2003/47.html
 Retrieved: http://www.bailii.org/uk/cases/UKHL/2001/52.html
 Retrieved: http://www.bailii.org/uk/cases/UKHL/2000/4.html
 Retrieved: http://www.bailii.org/uk/cases/UKHL/1954/AC/66.html
 Retrieved: http:/

Attempt 2

In [ ]:
#Attempt 2
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

def fetch_bailii_case_clean(url):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    try:
        resp = requests.get(url, headers=headers, timeout=20)
        if resp.status_code != 200:
            return {"Title": "Not Found", "Text": "", "Source": "Case Law", "URL": url}

        soup = BeautifulSoup(resp.text, 'html.parser')

        title = soup.title.get_text(strip=True) if soup.title else "No Title Found"
        title = re.sub(r'\s+\[.*?\] BAILII$', '', title)

        content = "\n\n".join(tag.get_text(" ", strip=True) for tag in soup.select("pre"))
        if not content:
            content = "\n\n".join(tag.get_text(" ", strip=True) for tag in soup.select("body"))

        return {
            "Title": title,
            "Text": content if content else "No text extracted",
            "Source": "Case Law",
            "URL": url
        }

    except Exception as e:
        return {"Title": "Error", "Text": str(e), "Source": "Case Law", "URL": url}

# List of 40+ key BAILII case URLs (contract and tort law)
bailii_urls = [
    # Tort (duty, negligence, liability)
    "http://www.bailii.org/uk/cases/UKHL/1932/AC/562.html",  # Donoghue v Stevenson
    "http://www.bailii.org/uk/cases/UKHL/1932/2.html",       # WN Hillas & Co Ltd v Arcos Ltd
    "http://www.bailii.org/uk/cases/UKHL/1965/1.html",       # Rylands v Fletcher
    "http://www.bailii.org/uk/cases/UKHL/1963/1.html",       # Hedley Byrne & Co Ltd v Heller & Partners Ltd
    "http://www.bailii.org/uk/cases/UKHL/1975/5.html",       # Anns v Merton London Borough Council
    "http://www.bailii.org/uk/cases/UKHL/1954/AC/665.html",  # Bolam v Friern Hospital Management Comm.
    "http://www.bailii.org/uk/cases/UKHL/1987/241.html",     # Smith v Littlewoods Organisation Ltd
    "http://www.bailii.org/uk/cases/UKHL/1985/21.html",      # Caparo Industries plc v Dickman
    "http://www.bailii.org/uk/cases/UKHL/1985/45.html",      # Spartan Steel & Alloys Ltd v Martin & Co
    "http://www.bailii.org/uk/cases/UKHL/1990/1.html",       # Home Office v Dorset Yacht Co Ltd
    "http://www.bailii.org/uk/cases/UKHL/1997/22.html",      # Fairchild v Glenhaven Funeral Services Ltd
    "http://www.bailii.org/uk/cases/UKHL/2002/22.html",      # McGhee v National Coal Board
    "http://www.bailii.org/uk/cases/UKHL/2003/47.html",      # Tomlinson v Congleton Borough Council
    "http://www.bailii.org/uk/cases/UKHL/2001/52.html",      # Chester v Afshar
    "http://www.bailii.org/uk/cases/UKHL/2000/4.html",       # Hunter v Canary Wharf Ltd
    "http://www.bailii.org/uk/cases/UKHL/1954/AC/66.html",   # Scott v London & St Katherine's Docks Co
    "http://www.bailii.org/uk/cases/UKHL/1964/1.html",       # Rookes v Barnard
    "http://www.bailii.org/uk/cases/UKHL/1954/2.html",       # Roe v Ministry of Health
    "http://www.bailii.org/uk/cases/UKHL/1961/405.html",     # Smith v Leech Brain & Co Ltd
    "http://www.bailii.org/uk/cases/UKHL/2003/61.html",      # Transco plc v Stockport Metropolitan Borough Council
    "http://www.bailii.org/uk/cases/UKHL/2001/22.html",      # Lister v Hesley Hall Ltd
    "http://www.bailii.org/uk/cases/UKHL/2006/17.html",      # Sienkiewicz v Greif (UK) Ltd
    "http://www.bailii.org/uk/cases/UKHL/1990/1.html",       # Sidaway v Board of Governors of the Bethlem Royal Hospital
    "http://www.bailii.org/uk/cases/UKHL/2003/52.html",      # Rees v Darlington Memorial Hospital NHS Trust
    "http://www.bailii.org/uk/cases/UKHL/1996/52.html",      # Alcock v Chief Constable of South Yorkshire Police
    "http://www.bailii.org/uk/cases/UKHL/1965/241.html",     # St Helen’s Smelting Co v Tipping
    "http://www.bailii.org/uk/cases/UKHL/2006/15.html",      # Sutradhar v Natural Environment Research Council
    "http://www.bailii.org/uk/cases/UKHL/1995/296.html",     # Spring v Guardian Assurance plc
    "http://www.bailii.org/uk/cases/UKHL/2008/17.html",      # Barclays Bank plc v O'Brien
    "http://www.bailii.org/uk/cases/UKHL/1993/40.html",      # Attorney-General v Blake
    "http://www.bailii.org/uk/cases/UKHL/1979/6.html",       # Gibson v Manchester City Council
    "http://www.bailii.org/uk/cases/UKHL/1998/10.html",      # South Australian Asset Management Corp v York Montagu Ltd
    "http://www.bailii.org/uk/cases/UKHL/1970/147.html",     # Bell v Lever Brothers Ltd
    "http://www.bailii.org/uk/cases/UKHL/1972/3.html",       # Walford v Miles
    "http://www.bailii.org/uk/cases/UKHL/1934/1.html",       # Southern Foundries (1926) Ltd v Shirlaw
    "http://www.bailii.org/uk/cases/UKHL/1990/1.html",       # Chester v Afshar (bookended)
    "http://www.bailii.org/uk/cases/UKHL/1992/1.html",       # Gillespie Bros v Roy Bowles Ltd
    "http://www.bailii.org/uk/cases/UKHL/2012/15.html",      # Stannard v Gore
    "http://www.bailii.org/uk/cases/UKHL/2003/47.html",      # Tomlinson v Congleton Borough Council (duplicate)
    "http://www.bailii.org/uk/cases/UKHL/2000/4.html"        # Hunter v Canary Wharf Ltd (duplicate)
#additional
    # Tort law cases
    "https://www.bailii.org/uk/cases/UKHL/1932/AC/562.html",  # Donoghue v Stevenson (1932)
    "https://www.bailii.org/uk/cases/UKHL/1975/5.html",       # Anns v Merton LBC (1975)
    "https://www.bailii.org/uk/cases/UKHL/1865/1.html",       # Rylands v Fletcher (1865)
    "https://www.bailii.org/uk/cases/UKHL/1963/1.html",       # Hedley Byrne v Heller (1963)
    "https://www.bailii.org/uk/cases/UKHL/2002/22.html",      # Fairchild v Glenhaven (2002)
    "https://www.bailii.org/uk/cases/UKHL/1973/1.html",       # McGhee v National Coal Board (1973)
    "https://www.bailii.org/uk/cases/UKHL/1957/1.html",       # Bolam v Friern Hospital Management Comm (1957)
    "https://www.bailii.org/uk/cases/UKHL/1951/1.html",       # Bolton v Stone (1951)
    "https://www.bailii.org/uk/cases/UKHL/1949/6.html",       # Barnett v Chelsea & Kensington Hospital (1949)
    "https://www.bailii.org/uk/cases/UKHL/1962/100.html",     # Smith v Leech Brain & Co Ltd (1962)
    "https://www.bailii.org/uk/cases/UKHL/1868/3.html",       # Rylands v Fletcher (1868 appeal)
    "https://www.bailii.org/uk/cases/UKHL/2003/52.html",      # Rees v Darlington Memorial Hospital (2003)
    "https://www.bailii.org/uk/cases/UKHL/1965/241.html",     # St Helen’s Smelting Co v Tipping (1865)
    "https://www.bailii.org/uk/cases/UKHL/1987/241.html",     # Smith v Littlewoods (1987)
    "https://www.bailii.org/uk/cases/UKHL/2003/61.html",      # Transco v Stockport MBC (2003)
    "https://www.bailii.org/uk/cases/UKHL/1990/1.html",       # Home Office v Dorset Yacht Co (1970)
    "https://www.bailii.org/uk/cases/UKHL/1970/147.html",     # Bell v Lever Brothers (1970)
    "https://www.bailii.org/uk/cases/UKHL/1964/1.html",       # Rookes v Barnard (1964)
    "https://www.bailii.org/uk/cases/UKHL/2001/22.html",      # Lister v Hesley Hall (2001)
    "https://www.bailii.org/uk/cases/UKHL/2006/17.html",      # Sienkiewicz v Greif (2006)

    # Contract law cases
    "https://www.bailii.org/uk/cases/UKHL/1893/QB/484.html",  # Carlill v Carbolic Smoke Ball Co (1893)
    "https://www.bailii.org/uk/cases/UKHL/1919/2KB/571.html",  # Balfour v Balfour (1919)
    "https://www.bailii.org/uk/cases/UKHL/1854/Exch/341.html", # Hadley v Baxendale (1854)
    "https://www.bailii.org/uk/cases/UKHL/1862/11CB/869.html", # Felthouse v Bindley (1862)
    "https://www.bailii.org/uk/cases/UKHL/1864/2HandC/906.html",# Raffles v Wichelhaus (1864)
    "https://www.bailii.org/uk/cases/UKHL/1960/ChD/2.html",    # Pharmaceutical Society v Boots (1953)
    "https://www.bailii.org/uk/cases/UKHL/1960/ChD/2.html",    # Chappell & Co Ltd v Nestle (1960)
    "https://www.bailii.org/uk/cases/UKHL/1968/2Ch/254.html",  # Partridge v Crittenden (1968)
    "https://www.bailii.org/uk/cases/UKHL/1818/AdamsLindsell.html", # Adams v Lindsell (1818)
    "https://www.bailii.org/uk/cases/UKHL/1876/DickinsonDodds.html", # Dickinson v Dodds (1876)
    "https://www.bailii.org/uk/cases/UKHL/1831/CollinsGodefrey.html",  # Collins v Godefrey (1831)
    "https://www.bailii.org/uk/cases/UKHL/1960/fisherbell.html",      # Fisher v Bell (1960)
    "https://www.bailii.org/uk/cases/UKHL/1979/GibsonManchester.html",# Gibson v Manchester CC (1979)
    "https://www.bailii.org/uk/cases/UKHL/1964/LiverpoolIrwin.html",  # Liverpool City Council v Irwin (1977)
    "https://www.bailii.org/uk/cases/UKHL/1980/WoodarWimpey.html",     # Woodar v Wimpey (1980)
    "https://www.bailii.org/uk/cases/UKHL/2000/ParadineJane.html",     # Paradine v Jane (1602)
    "https://www.bailii.org/uk/cases/UKHL/1961/ScruttonsMidland.html",# Scruttons Ltd v Midland Silicones (1961)
    "https://www.bailii.org/uk/cases/UKHL/1992/MurphyBrentwood.html",  # Murphy v Brentwood (1992)
    "https://www.bailii.org/uk/cases/UKHL/1961/BoltonStone.html",     # Bolton v Stone (contract context)
    "https://www.bailii.org/uk/cases/UKHL/1987/WilsherEssex.html",    # Wilsher v Essex AHA (1987)
    "https://www.bailii.org/uk/cases/UKHL/2004/ChesterAfshar.html",   # Chester v Afshar (2004)
    "https://www.bailii.org/uk/cases/UKHL/1999/HunterCanary.html",   # Hunter v Canary Wharf (1999)
    "https://www.bailii.org/uk/cases/UKHL/2008/BarclaysOBrien.html",  # Barclays Bank plc v O'Brien (2008)
    "https://www.bailii.org/uk/cases/UKHL/1968/TaylorFashions.html"   # Taylor Fashions v Liverpool Victoria (1980)
]


results = []
for i, url in enumerate(bailii_urls, 1):
    case_data = fetch_bailii_case_clean(url)
    print(f"[{i}/{len(bailii_urls)}] Retrieved: {case_data['Title']}")
    results.append(case_data)
    time.sleep(3)

df = pd.DataFrame(results)
df.to_csv("BAILII_Case_Law_Formatted.csv", index=False)
df.head()


[1/83] Retrieved: Access denied
[2/83] Retrieved: Access denied
[3/83] Retrieved: Access denied
[4/83] Retrieved: Access denied
[5/83] Retrieved: Access denied
[6/83] Retrieved: Access denied
[7/83] Retrieved: Access denied
[8/83] Retrieved: Access denied
[9/83] Retrieved: Access denied
[10/83] Retrieved: Access denied
[11/83] Retrieved: Access denied
[12/83] Retrieved: Access denied
[13/83] Retrieved: Access denied
[14/83] Retrieved: Access denied
[15/83] Retrieved: Access denied
[16/83] Retrieved: Access denied
[17/83] Retrieved: Access denied
[18/83] Retrieved: Access denied
[19/83] Retrieved: Access denied
[20/83] Retrieved: Access denied
[21/83] Retrieved: Access denied
[22/83] Retrieved: Access denied
[23/83] Retrieved: Access denied
[24/83] Retrieved: Access denied
[25/83] Retrieved: Access denied
[26/83] Retrieved: Access denied
[27/83] Retrieved: Access denied
[28/83] Retrieved: Access denied
[29/83] Retrieved: Access denied
[30/83] Retrieved: Access denied
[31/83] Retrieved: 

,Title,Text,Source,URL
0,Access denied,Your system has been denied access to BAILII f...,Case Law,http://www.bailii.org/uk/cases/UKHL/1932/AC/56...
1,Access denied,Your system has been denied access to BAILII f...,Case Law,http://www.bailii.org/uk/cases/UKHL/1932/2.html
2,Access denied,Your system has been denied access to BAILII f...,Case Law,http://www.bailii.org/uk/cases/UKHL/1965/1.html
3,Access denied,Your system has been denied access to BAILII f...,Case Law,http://www.bailii.org/uk/cases/UKHL/1963/1.html
4,Access denied,Your system has been denied access to BAILII f...,Case Law,http://www.bailii.org/uk/cases/UKHL/1975/5.html


adding more BAILII cases (for future reference): bailii_urls = [
    # Tort law cases
    "https://www.bailii.org/uk/cases/UKHL/1932/AC/562.html",  # Donoghue v Stevenson (1932)
    "https://www.bailii.org/uk/cases/UKHL/1975/5.html",       # Anns v Merton LBC (1975)
    "https://www.bailii.org/uk/cases/UKHL/1865/1.html",       # Rylands v Fletcher (1865)
    "https://www.bailii.org/uk/cases/UKHL/1963/1.html",       # Hedley Byrne v Heller (1963)
    "https://www.bailii.org/uk/cases/UKHL/2002/22.html",      # Fairchild v Glenhaven (2002)
    "https://www.bailii.org/uk/cases/UKHL/1973/1.html",       # McGhee v National Coal Board (1973)
    "https://www.bailii.org/uk/cases/UKHL/1957/1.html",       # Bolam v Friern Hospital Management Comm (1957)
    "https://www.bailii.org/uk/cases/UKHL/1951/1.html",       # Bolton v Stone (1951)
    "https://www.bailii.org/uk/cases/UKHL/1949/6.html",       # Barnett v Chelsea & Kensington Hospital (1949)
    "https://www.bailii.org/uk/cases/UKHL/1962/100.html",     # Smith v Leech Brain & Co Ltd (1962)
    "https://www.bailii.org/uk/cases/UKHL/1868/3.html",       # Rylands v Fletcher (1868 appeal)
    "https://www.bailii.org/uk/cases/UKHL/2003/52.html",      # Rees v Darlington Memorial Hospital (2003)
    "https://www.bailii.org/uk/cases/UKHL/1965/241.html",     # St Helen’s Smelting Co v Tipping (1865)
    "https://www.bailii.org/uk/cases/UKHL/1987/241.html",     # Smith v Littlewoods (1987)
    "https://www.bailii.org/uk/cases/UKHL/2003/61.html",      # Transco v Stockport MBC (2003)
    "https://www.bailii.org/uk/cases/UKHL/1990/1.html",       # Home Office v Dorset Yacht Co (1970)
    "https://www.bailii.org/uk/cases/UKHL/1970/147.html",     # Bell v Lever Brothers (1970)
    "https://www.bailii.org/uk/cases/UKHL/1964/1.html",       # Rookes v Barnard (1964)
    "https://www.bailii.org/uk/cases/UKHL/2001/22.html",      # Lister v Hesley Hall (2001)
    "https://www.bailii.org/uk/cases/UKHL/2006/17.html",      # Sienkiewicz v Greif (2006)
    
    # Contract law cases
    "https://www.bailii.org/uk/cases/UKHL/1893/QB/484.html",  # Carlill v Carbolic Smoke Ball Co (1893)
    "https://www.bailii.org/uk/cases/UKHL/1919/2KB/571.html",  # Balfour v Balfour (1919)
    "https://www.bailii.org/uk/cases/UKHL/1854/Exch/341.html", # Hadley v Baxendale (1854)
    "https://www.bailii.org/uk/cases/UKHL/1862/11CB/869.html", # Felthouse v Bindley (1862)
    "https://www.bailii.org/uk/cases/UKHL/1864/2HandC/906.html",# Raffles v Wichelhaus (1864)
    "https://www.bailii.org/uk/cases/UKHL/1960/ChD/2.html",    # Pharmaceutical Society v Boots (1953)
    "https://www.bailii.org/uk/cases/UKHL/1960/ChD/2.html",    # Chappell & Co Ltd v Nestle (1960)
    "https://www.bailii.org/uk/cases/UKHL/1968/2Ch/254.html",  # Partridge v Crittenden (1968)
    "https://www.bailii.org/uk/cases/UKHL/1818/AdamsLindsell.html", # Adams v Lindsell (1818)
    "https://www.bailii.org/uk/cases/UKHL/1876/DickinsonDodds.html", # Dickinson v Dodds (1876)
    "https://www.bailii.org/uk/cases/UKHL/1831/CollinsGodefrey.html",  # Collins v Godefrey (1831)
    "https://www.bailii.org/uk/cases/UKHL/1960/fisherbell.html",      # Fisher v Bell (1960)
    "https://www.bailii.org/uk/cases/UKHL/1979/GibsonManchester.html",# Gibson v Manchester CC (1979)
    "https://www.bailii.org/uk/cases/UKHL/1964/LiverpoolIrwin.html",  # Liverpool City Council v Irwin (1977)
    "https://www.bailii.org/uk/cases/UKHL/1980/WoodarWimpey.html",     # Woodar v Wimpey (1980)
    "https://www.bailii.org/uk/cases/UKHL/2000/ParadineJane.html",     # Paradine v Jane (1602)
    "https://www.bailii.org/uk/cases/UKHL/1961/ScruttonsMidland.html",# Scruttons Ltd v Midland Silicones (1961)
    "https://www.bailii.org/uk/cases/UKHL/1992/MurphyBrentwood.html",  # Murphy v Brentwood (1992)
    "https://www.bailii.org/uk/cases/UKHL/1961/BoltonStone.html",     # Bolton v Stone (contract context)
    "https://www.bailii.org/uk/cases/UKHL/1987/WilsherEssex.html",    # Wilsher v Essex AHA (1987)
    "https://www.bailii.org/uk/cases/UKHL/2004/ChesterAfshar.html",   # Chester v Afshar (2004)
    "https://www.bailii.org/uk/cases/UKHL/1999/HunterCanary.html",   # Hunter v Canary Wharf (1999)
    "https://www.bailii.org/uk/cases/UKHL/2008/BarclaysOBrien.html",  # Barclays Bank plc v O'Brien (2008)
    "https://www.bailii.org/uk/cases/UKHL/1968/TaylorFashions.html"   # Taylor Fashions v Liverpool Victoria (1980)
]


## phase 4 Add Government & NGO Legal Advice

scrape and parse content from reliable, publicly available sources such as:

Citizens Advice (https://www.citizensadvice.org.uk/)

Gov.uk justice guides (https://www.gov.uk/browse/justice)

(Optional later: Law Society, Shelter, ACAS, etc.)

In [ ]:
#Phase 4: Govt & NGO Legal advice
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_citizens_advice_page(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        r = requests.get(url, headers=headers, timeout=10)
        if r.status_code != 200:
            return None
        soup = BeautifulSoup(r.text, "html.parser")
        title = soup.title.text.strip()
        paragraphs = soup.select("div.content__body p")
        text = "\n\n".join([p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)])
        return {
            "Title": title,
            "Text": text,
            "Source": "Citizens Advice",
            "URL": url
        }
    except Exception as e:
        return {"Title": "Error", "Text": str(e), "Source": "Citizens Advice", "URL": url}

In [ ]:

#URL list
ca_urls = [
    # Housing & Tenancy
    "https://www.citizensadvice.org.uk/housing/renting-privately/during-your-tenancy/getting-repairs-done/",
    "https://www.citizensadvice.org.uk/debt-and-money/rent-arrears/",
    "https://www.citizensadvice.org.uk/housing/renting-privately/during-your-tenancy/eviction-for-rent-arrears/",
    "https://www.citizensadvice.org.uk/housing/renting-privately/",
    "https://www.citizensadvice.org.uk/housing/homelessness/applying-for-homeless-help/",

    # Personal Injury & Accidents
    "https://www.citizensadvice.org.uk/law-and-courts/personal-injuries/",
    "https://www.citizensadvice.org.uk/law-and-courts/personal-injuries/making-a-personal-injury-claim/",
    "https://www.citizensadvice.org.uk/health/nhs-healthcare/complaining-about-nhs-care/",

    # Consumer Rights
    "https://www.citizensadvice.org.uk/consumer/somethings-gone-wrong-with-a-purchase/return-faulty-goods/",
    "https://www.citizensadvice.org.uk/consumer/your-rights-when-you-buy-goods/",
    "https://www.citizensadvice.org.uk/consumer/changed-your-mind/cancelling-a-service-youve-arranged/",

    # Employment Rights
    "https://www.citizensadvice.org.uk/work/rights-at-work/time-off-work/",
    "https://www.citizensadvice.org.uk/work/sick-pay/",
    "https://www.citizensadvice.org.uk/work/leaving-a-job/dismissal/check-if-your-dismissal-is-unfair/",
    "https://www.citizensadvice.org.uk/work/leaving-a-job/redundancy/",

    # Family & Relationships
    "https://www.citizensadvice.org.uk/family/ending-a-relationship/",
    "https://www.citizensadvice.org.uk/family/ending-a-relationship/when-you-separate/dividing-money-and-belongings/",
    "https://www.citizensadvice.org.uk/family/gender-violence/domestic-violence-and-abuse/getting-an-injunction/",

    # Police, Crime & Courts
    "https://www.citizensadvice.org.uk/law-and-courts/legal-system/police/police-powers/",
    "https://www.citizensadvice.org.uk/law-and-courts/legal-system/going-to-court-without-a-solicitor/",
    "https://www.citizensadvice.org.uk/law-and-courts/legal-system/police/police-powers/your-rights-when-being-questioned/",

    # GOV.UK Legal
    "https://www.gov.uk/how-to-resolve-neighbour-disputes",
    "https://www.gov.uk/complain-about-solicitor",
    "https://www.gov.uk/check-legal-aid",
    "https://www.gov.uk/employment-tribunals"
]

# List to collect scraped advice data
advice_entries = []

# Loop through URLs with error checks
for i, url in enumerate(ca_urls, 1):
    data = scrape_citizens_advice_page(url)

    if data is None:
        print(f"[{i}/{len(ca_urls)}] Failed to fetch: {url}")
        continue

    if not isinstance(data, dict) or 'Title' not in data or 'Text' not in data:
        print(f"[{i}/{len(ca_urls)}]  Malformed data from: {url}")
        continue

    print(f"[{i}/{len(ca_urls)}] Fetched: {data['Title']}")
    advice_entries.append(data)
    time.sleep(2)  # polite delay to avoid overloading the site

# Save to CSV
df_advice = pd.DataFrame(advice_entries)
df_advice.to_csv("gov_ngo_advice.csv", index=False)


[1/25] Failed to fetch: https://www.citizensadvice.org.uk/housing/renting-privately/during-your-tenancy/getting-repairs-done/
[2/25] Fetched: Rent arrears - Citizens Advice
[3/25] Failed to fetch: https://www.citizensadvice.org.uk/housing/renting-privately/during-your-tenancy/eviction-for-rent-arrears/
[4/25] Fetched: Housing - Citizens Advice
[5/25] Failed to fetch: https://www.citizensadvice.org.uk/housing/homelessness/applying-for-homeless-help/
[6/25] Fetched: Claiming compensation for a personal injury - Citizens Advice
[7/25] Failed to fetch: https://www.citizensadvice.org.uk/law-and-courts/personal-injuries/making-a-personal-injury-claim/
[8/25] Failed to fetch: https://www.citizensadvice.org.uk/health/nhs-healthcare/complaining-about-nhs-care/
[9/25] Fetched: Return faulty goods - Citizens Advice
[10/25] Failed to fetch: https://www.citizensadvice.org.uk/consumer/your-rights-when-you-buy-goods/
[11/25] Fetched: Cancelling a service you’ve arranged - Citizens Advice
[12/25] Fail

## Phase 5 - scraping Supreme Court case HTML

Use structured page:

https://www.supremecourt.uk/decided-cases/

parse each case page for the summary and full judgment (HTML).

In [ ]:
#Phase 5: Supreme Court judgements
def scrape_uksc_case(url):
    try:
        r = requests.get(url, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        title = soup.find("h1").text.strip()
        summary = soup.select_one(".entry-content").get_text(" ", strip=True)
        return {
            "Title": title,
            "Text": summary,
            "Source": "UK Supreme Court",
            "URL": url
        }
    except Exception as e:
        return {"Title": "Error", "Text": str(e), "Source": "UK Supreme Court", "URL": url}

!pip install pymupdf

import time
import requests
import fitz  # PyMuPDF
import pandas as pd

# List of Supreme Court PDF judgment URLs
uksc_urls = [
    # Tort / Negligence
    "https://www.supremecourt.uk/cases/docs/uksc-2018-004_judgment.pdf",
    "https://www.supremecourt.uk/cases/docs/uksc-2015-011_judgment.pdf",
    "https://www.supremecourt.uk/cases/docs/uksc-2015-016_judgment.pdf",
    "https://www.supremecourt.uk/cases/docs/uksc-2014-007_judgment.pdf",
    # Contract / Commercial
    "https://www.supremecourt.uk/cases/docs/uksc-2015-0193_judgment.pdf",
    "https://www.supremecourt.uk/cases/docs/uksc-2015-0021_judgment.pdf",
    "https://www.supremecourt.uk/cases/docs/uksc-2015-0043_judgment.pdf",
    "https://www.supremecourt.uk/cases/docs/uksc-2015-0058_judgment.pdf",
    # Human rights tort + duty of care
    "https://www.supremecourt.uk/cases/docs/uksc-2016-0196-press-summary.pdf",
    # Mixed tort/contract/duty
    "https://www.supremecourt.uk/cases/docs/uksc-2012-0057_judgment.pdf",
    "https://www.supremecourt.uk/cases/docs/uksc-2012-0229_judgment.pdf",
    "https://www.supremecourt.uk/cases/docs/uksc-2013-0193_judgment.pdf",
    "https://www.supremecourt.uk/cases/docs/uksc-2012-0276_judgment.pdf",
    "https://www.supremecourt.uk/cases/docs/uksc-2013-0049_judgment.pdf",
    "https://www.supremecourt.uk/cases/docs/uksc-2014-0158_judgment.pdf"
]

def scrape_uksc_pdf_case(url):
    try:
        response = requests.get(url, timeout=20)
        if response.status_code != 200:
            return {"Title": "Error", "Text": f"HTTP {response.status_code}", "URL": url, "Source": "UK Supreme Court"}

        with fitz.open(stream=response.content, filetype="pdf") as doc:
            text = "\n".join(page.get_text() for page in doc)

        # Basic title fallback: use filename or first line
        base_title = url.split("/")[-1].replace(".pdf", "").replace("_", " ").title()
        first_line = text.strip().split('\n')[0][:200] if text.strip() else base_title

        return {
            "Title": first_line,
            "Text": text.strip(),
            "URL": url,
            "Source": "UK Supreme Court"
        }

    except Exception as e:
        return {"Title": "Error", "Text": str(e), "URL": url, "Source": "UK Supreme Court"}

# Loop through all URLs
uksc_cases = []
for i, url in enumerate(uksc_urls, 1):
    data = scrape_uksc_pdf_case(url)
    print(f"[{i}/{len(uksc_urls)}] Fetched: {data['Title']}")
    uksc_cases.append(data)
    time.sleep(2)

# Create DataFrame and save
df_uksc = pd.DataFrame(uksc_cases)
df_uksc.to_csv("supreme_court_cases.csv", index=False)
print("Saved to supreme_court_cases.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 75.8 MB/s eta 0:00:00
[1/15] Fetched: Visit JCPC
[2/15] Fetched: Visit JCPC
[3/15] Fetched: Visit JCPC
[4/15] Fetched: Visit JCPC
[5/15] Fetched: Visit JCPC
[6/15] Fetched: Visit JCPC
[7/15] Fetched: Visit JCPC
[8/15] Fetched: Visit JCPC
[9/15] Fetched: Visit JCPC
[10/15] Fetched: Visit JCPC
[11/15] Fetched: Visit JCPC
[12/15] Fetched: Visit JCPC
[13/15] Fetched: Visit JCPC
[14/15] Fetched: Visit JCPC
[15/15] Fetched: Visit JCPC
Saved to supreme_court_cases.csv


### Removing rows with No Title Found No text extracted
add domain column (tort or contract)

In [ ]:
import pandas as pd

# Load CSVs
df_statutes = pd.read_csv("uk_statutes_extracted.csv")
df_cases = pd.read_csv("uk_case_law_corpus_hybrid.csv")
df_bailii = pd.read_csv("BAILII_Case_Law_Formatted.csv")
df_NGO = pd.read_csv("gov_ngo_advice.csv")
df_uksc = pd.read_csv("supreme_court_cases.csv")

# Add Dataset Source Tag
df_statutes["Dataset"] = "Legislation.gov.uk"
df_cases["Dataset"] = "LawTeacher/eLawResources"
df_bailii["Dataset"] = "BAILII.org"
df_NGO["Dataset"] = "Gov.uk"
df_uksc["Dataset"] = "Supreme Court"

# Ensure all have 'Source' and add 'LegalDomain'
for df in [df_statutes, df_cases, df_bailii]:
    if "Source" not in df.columns:
        df["Source"] = "Unknown"
    df["LegalDomain"] = "Unknown"  # can annotate later as 'Tort', 'Contract', etc.

# Combine All
df_all = pd.concat([df_statutes, df_cases, df_bailii], ignore_index=True)

# Clean Rows
df_all_cleaned = df_all[
    (df_all["Title"].notna()) &
    (df_all["Text"].notna()) &
    (df_all["Title"].str.strip().str.lower() != "no title found") &
    (df_all["Text"].str.strip().str.lower() != "no text extracted") &
    (df_all["Title"].str.strip() != "") &
    (df_all["Text"].str.strip() != "")
]

# Save
df_all_cleaned.to_csv("legal_corpus_combined_cleaned.csv", index=False)

# Summary
print("Final cleaned dataset shape:", df_all_cleaned.shape)
print(df_all_cleaned.tail())

Final cleaned dataset shape: (121, 6)
             Title                                               Text  \
137  Access denied  Your system has been denied access to BAILII f...   
138  Access denied  Your system has been denied access to BAILII f...   
139  Access denied  Your system has been denied access to BAILII f...   
140  Access denied  Your system has been denied access to BAILII f...   
141  Access denied  Your system has been denied access to BAILII f...   

       Source     Dataset LegalDomain  \
137  Case Law  BAILII.org     Unknown   
138  Case Law  BAILII.org     Unknown   
139  Case Law  BAILII.org     Unknown   
140  Case Law  BAILII.org     Unknown   
141  Case Law  BAILII.org     Unknown   

                                                   URL  
137  https://www.bailii.org/uk/cases/UKHL/1987/Wils...  
138  https://www.bailii.org/uk/cases/UKHL/2004/Ches...  
139  https://www.bailii.org/uk/cases/UKHL/1999/Hunt...  
140  https://www.bailii.org/uk/cases/UKHL/2008/B

In [ ]:
df.head()

,Title,Text,Source,URL,Dataset,LegalDomain
0,Access denied,Your system has been denied access to BAILII f...,Case Law,http://www.bailii.org/uk/cases/UKHL/1932/AC/56...,BAILII.org,Unknown
1,Access denied,Your system has been denied access to BAILII f...,Case Law,http://www.bailii.org/uk/cases/UKHL/1932/2.html,BAILII.org,Unknown
2,Access denied,Your system has been denied access to BAILII f...,Case Law,http://www.bailii.org/uk/cases/UKHL/1965/1.html,BAILII.org,Unknown
3,Access denied,Your system has been denied access to BAILII f...,Case Law,http://www.bailii.org/uk/cases/UKHL/1963/1.html,BAILII.org,Unknown
4,Access denied,Your system has been denied access to BAILII f...,Case Law,http://www.bailii.org/uk/cases/UKHL/1975/5.html,BAILII.org,Unknown


Using regex to identify legal domain as showing as 'unknown'
tort-related keywords (like "duty" or "liability") overlap with contract law, especially in hybrid cases.

### Revised Legal Domain Classifier (Mutual Exclusion + Confidence Scoring)

In [ ]:
import pandas as pd
import re

# Load data
df = pd.read_csv("legal_corpus_combined_cleaned.csv")

# Domain keyword scoring system
keyword_scores = {
    "Contract": [
        "contract", "breach", "offer", "acceptance", "consideration",
        "terms", "frustration", "misrepresentation", "invitation to treat",
        "promissory estoppel", "agreement"
    ],
    "Tort": [
        "negligence", "duty of care", "liability", "causation", "damage",
        "remoteness", "occupier", "trespass", "nuisance", "personal injury",
        "vicarious", "defamation"
    ],
    "Employment": [
        "employee", "employer", "dismissal", "redundancy", "tribunal",
        "discrimination", "wages", "employment rights"
    ],
    "Consumer": [
        "consumer", "goods", "trader", "unfair", "product", "merchantable"
    ],
    "Property": [
        "lease", "tenancy", "land", "mortgage", "easement", "freehold", "title"
    ]
}

# Scoring function
def score_domains(text):
    scores = {domain: 0 for domain in keyword_scores}
    text = text.lower()
    for domain, keywords in keyword_scores.items():
        for kw in keywords:
            if re.search(rf'\b{kw}\b', text):
                scores[domain] += 1
    max_score = max(scores.values())
    if max_score == 0:
        return "Unknown"
    top_domains = [d for d, s in scores.items() if s == max_score]
    return top_domains[0] if len(top_domains) == 1 else "Mixed"

# Apply to dataset
df["LegalDomain"] = df.apply(lambda row: score_domains(f"{row['Title']} {row['Text']}"), axis=1)

# Save output
df.to_csv("legal_corpus_domain_tagged.csv", index=False)

# Show summary
print("Legal domain tagging complete.")
print(df["LegalDomain"].value_counts())


Legal domain tagging complete.
LegalDomain
Tort          89
Contract      18
Mixed          6
Employment     4
Property       2
Unknown        2
Name: count, dtype: int64


In [ ]:
df.head()

,Title,Text,Source,Dataset,LegalDomain,URL
0,Automated and Electric Vehicles Act 2018,"B\n\nThe Secretary of State must prepare, and ...",Statute,Legislation.gov.uk,Tort,NaN
1,Unfair Contract Terms Act 1977,"For the purposes of this Part of this Act, “\n...",Statute,Legislation.gov.uk,Contract,NaN
2,Sale of Goods Act 1979,This Act applies to contracts of sale of goods...,Statute,Legislation.gov.uk,Contract,NaN
3,Consumer Rights Act 2015,B\n\nThis Part applies where there is an agree...,Statute,Legislation.gov.uk,Contract,NaN
4,Employment Rights Act 1996,Be it enacted by the Queen’s most Excellent Ma...,Statute,Legislation.gov.uk,Employment,NaN


In [ ]:
df.tail()

,Title,Text,Source,Dataset,LegalDomain,URL
116,Access denied,Your system has been denied access to BAILII f...,Case Law,BAILII.org,Tort,https://www.bailii.org/uk/cases/UKHL/1987/Wils...
117,Access denied,Your system has been denied access to BAILII f...,Case Law,BAILII.org,Tort,https://www.bailii.org/uk/cases/UKHL/2004/Ches...
118,Access denied,Your system has been denied access to BAILII f...,Case Law,BAILII.org,Tort,https://www.bailii.org/uk/cases/UKHL/1999/Hunt...
119,Access denied,Your system has been denied access to BAILII f...,Case Law,BAILII.org,Tort,https://www.bailii.org/uk/cases/UKHL/2008/Barc...
120,Access denied,Your system has been denied access to BAILII f...,Case Law,BAILII.org,Tort,https://www.bailii.org/uk/cases/UKHL/1968/Tayl...


Merging with all datasets

In [ ]:
import pandas as pd

# Load all datasets
statutes_df = pd.read_csv("uk_statutes_extracted.csv")
hybrid_cases_df = pd.read_csv("uk_case_law_corpus_hybrid.csv")
bailii_df = pd.read_csv("BAILII_Case_Law_Formatted.csv")
NGO_df = pd.read_csv("gov_ngo_advice.csv")
uksc_df = pd.read_csv("supreme_court_cases.csv")

# Step 1: Clean and normalize column names
def standardize_columns(df, dataset_name, data_type):
    df = df.rename(columns=lambda x: x.strip().capitalize())
    if 'Title' not in df.columns or 'Text' not in df.columns:
        raise ValueError(f"{dataset_name} missing required columns.")

    df['Type'] = data_type
    df['Source'] = dataset_name
    return df[['Title', 'Text', 'Type', 'Source']]

# Step 2: Apply standardization to each dataset
statutes_df = standardize_columns(statutes_df, "Legislation.gov.uk", "Statute")
hybrid_cases_df = standardize_columns(hybrid_cases_df, "Hybrid", "Case Law")
bailii_df = standardize_columns(bailii_df, "BAILII", "Case Law")

# Step 3: Combine all datasets
combined_df = pd.concat([statutes_df, hybrid_cases_df, bailii_df], ignore_index=True)

# Optional: Drop empty rows
combined_df.dropna(subset=["Title", "Text"], inplace=True)

# Step 4: Save to a new CSV
combined_df.to_csv("merged_legal_corpus.csv", index=False)

# Show result
print(" Merged dataset saved as 'merged_legal_corpus.csv'")
print(combined_df.sample(5))  # Show a few rows for verification


 Merged dataset saved as 'merged_legal_corpus.csv'
                                     Title  \
18   Local Government and Housing Act 1989   
65                           Access denied   
67                           Access denied   
109                          Access denied   
4               Employment Rights Act 1996   

                                                  Text      Type  \
18   A person shall be disqualified from becoming (...   Statute   
65   Your system has been denied access to BAILII f...  Case Law   
67   Your system has been denied access to BAILII f...  Case Law   
109  Your system has been denied access to BAILII f...  Case Law   
4    Be it enacted by the Queen’s most Excellent Ma...   Statute   

                 Source  
18   Legislation.gov.uk  
65               BAILII  
67               BAILII  
109              BAILII  
4    Legislation.gov.uk  


In [ ]:
df = pd.read_csv("merged_legal_corpus.csv")
df.head()

,Title,Text,Type,Source
0,Automated and Electric Vehicles Act 2018,"B\n\nThe Secretary of State must prepare, and ...",Statute,Legislation.gov.uk
1,Unfair Contract Terms Act 1977,"For the purposes of this Part of this Act, “\n...",Statute,Legislation.gov.uk
2,Sale of Goods Act 1979,This Act applies to contracts of sale of goods...,Statute,Legislation.gov.uk
3,Consumer Rights Act 2015,B\n\nThis Part applies where there is an agree...,Statute,Legislation.gov.uk
4,Employment Rights Act 1996,Be it enacted by the Queen’s most Excellent Ma...,Statute,Legislation.gov.uk


In [ ]:
df.tail()

,Title,Text,Type,Source
117,Access denied,Your system has been denied access to BAILII f...,Case Law,BAILII
118,Access denied,Your system has been denied access to BAILII f...,Case Law,BAILII
119,Access denied,Your system has been denied access to BAILII f...,Case Law,BAILII
120,Access denied,Your system has been denied access to BAILII f...,Case Law,BAILII
121,Access denied,Your system has been denied access to BAILII f...,Case Law,BAILII


More accurate differentiation between tort and contract.

Resolves overlapping keywords (e.g., "liability" in both domains).

Returns "Mixed" when both domains score equally — safer than guessing.



#SKIP AND RUN FROM HERE TO SAVE HASSLE OF SCRAPING
Merging again with uploaded presaved csv files in event that websites are down or unable to be html scraped

In [ ]:
!pip install --upgrade pandas

  Using cached pandas-3.0.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 111.6 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.3.3
    Uninstalling pandas-2.3.3:
      Successfully uninstalled pandas-2.3.3


In [ ]:
import pandas as pd
print(pd.__version__)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

# Load all datasets
statutes_df = pd.read_csv("uk_statutes_extracted.csv")
hybrid_cases_df = pd.read_csv("uk_case_law_corpus_hybrid.csv")
bailii_df = pd.read_csv("BAILII_Case_Law_Formatted.csv")
NGO_df = pd.read_csv("gov_ngo_advice.csv")
uksc_df = pd.read_csv("supreme_court_cases.csv")

# Step 1: Clean and normalise column names
def standardize_columns(df, dataset_name, data_type):
    df = df.rename(columns=lambda x: x.strip().capitalize())
    if 'Title' not in df.columns or 'Text' not in df.columns:
        raise ValueError(f"{dataset_name} missing required columns.")

    df['Type'] = data_type
    df['Source'] = dataset_name
    return df[['Title', 'Text', 'Type', 'Source']]

# Step 2: Apply standardisation to each dataset
statutes_df = standardize_columns(statutes_df, "Legislation.gov.uk", "Statute")
hybrid_cases_df = standardize_columns(hybrid_cases_df, "Hybrid", "Case Law")
bailii_df = standardize_columns(bailii_df, "BAILII", "Case Law")

# Step 3: Combine all datasets
combined_df = pd.concat([statutes_df, hybrid_cases_df, bailii_df], ignore_index=True)

# Optional: Drop empty rows
combined_df.dropna(subset=["Title", "Text"], inplace=True)

# Step 4: Save to a new CSV
combined_df.to_csv("merged_legal_corpus.csv", index=False)

# Show result
print(" Merged dataset saved as 'merged_legal_corpus.csv'")
print(combined_df.sample(5))  # Show a few rows for verification

In [ ]:
df = pd.read_csv("merged_legal_corpus.csv")
df.head()

In [ ]:
df.tail()

### Cleaning again

In [ ]:
import pandas as pd
from pathlib import Path

# 1) Load the single merged file from Step 1
MERGED = "merged_legal_corpus.csv"
if not Path(MERGED).exists():
    raise FileNotFoundError(
        f"Can't find {MERGED}. Run the merge step first to create it."
    )

# Read as strings to avoid dtype weirdness
df_all = pd.read_csv(MERGED, dtype=str, encoding_errors="replace", low_memory=False)

# Make sure key columns exist even if some sources didn't have them
for col in ["Title", "Text", "URI"]:
    if col not in df_all.columns:
        df_all[col] = ""

# 2) Clean rows: drop empties and obvious junk
BAD = {"no title found", "no text extracted", "nan", "none", "null", ""}

def ok(col):
    s = df_all[col].astype(str).str.strip().str.lower()
    return ~s.isin(BAD)

mask = ok("Title") & ok("Text")
df_all_cleaned = df_all[mask].copy()

# 3) De-duplicate (prefer using URI if present, otherwise Title+Text)
if "URI" in df_all_cleaned.columns and df_all_cleaned["URI"].astype(str).str.strip().any():
    df_all_cleaned = df_all_cleaned.drop_duplicates(subset=["URI", "Text"])
else:
    df_all_cleaned = df_all_cleaned.drop_duplicates(subset=["Title", "Text"])

# 4) Save cleaned outputs (one with old name, one the pipeline can use)
Path("data").mkdir(exist_ok=True)
df_all_cleaned.to_csv("legal_corpus_combined_cleaned.csv", index=False)  # original filename
df_all_cleaned.to_csv("data/legal_corpus_raw.csv", index=False)          # nice name for downstream steps

# 5) Summary
print("Final cleaned dataset shape:", df_all_cleaned.shape)
print(df_all_cleaned.tail(5))

In [ ]:
df.head()

### Revised Legal Domain Classifier (Mutual Exclusion + Confidence Scoring) - AGAIN AS NOW MERGED AND CLEANED

In [ ]:
import pandas as pd
import re

# Load data
df = pd.read_csv("legal_corpus_combined_cleaned.csv")

# Domain keyword scoring system
keyword_scores = {
    "Contract": [
        "contract", "breach", "offer", "acceptance", "consideration",
        "terms", "frustration", "misrepresentation", "invitation to treat",
        "promissory estoppel", "agreement"
    ],
    "Tort": [
        "negligence", "duty of care", "liability", "causation", "damage",
        "remoteness", "occupier", "trespass", "nuisance", "personal injury",
        "vicarious", "defamation"
    ],
    "Employment": [
        "employee", "employer", "dismissal", "redundancy", "tribunal",
        "discrimination", "wages", "employment rights"
    ],
    "Consumer": [
        "consumer", "goods", "trader", "unfair", "product", "merchantable"
    ],
    "Property": [
        "lease", "tenancy", "land", "mortgage", "easement", "freehold", "title"
    ]
}

# Scoring function
def score_domains(text):
    scores = {domain: 0 for domain in keyword_scores}
    text = text.lower()
    for domain, keywords in keyword_scores.items():
        for kw in keywords:
            if re.search(rf'\b{kw}\b', text):
                scores[domain] += 1
    max_score = max(scores.values())
    if max_score == 0:
        return "Unknown"
    top_domains = [d for d, s in scores.items() if s == max_score]
    return top_domains[0] if len(top_domains) == 1 else "Mixed"

# Apply to dataset
df["LegalDomain"] = df.apply(lambda row: score_domains(f"{row['Title']} {row['Text']}"), axis=1)

# Save output
df.to_csv("legal_corpus_domain_tagged.csv", index=False)

# Show summary
print("Legal domain tagging complete.")
print(df["LegalDomain"].value_counts())


In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, faiss
from sentence_transformers import SentenceTransformer

# Inputs/outputs
CSV_IN   = "legal_corpus_combined_cleaned.csv"   # produced by cleaning/tagging step
DATA_DIR = Path("data"); DATA_DIR.mkdir(parents=True, exist_ok=True)
DB_CSV   = DATA_DIR / "case_text_db.csv"         # normalized copy for search
IDX_PATH = DATA_DIR / "case_index.faiss"         # FAISS index on disk

# Retrieval model (pick one)
EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"   # higher quality (slower)
# EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # faster (good enough)

# quick sanity print to prove imports are good
import numpy, transformers, sentence_transformers
print("numpy:", numpy.__version__)
print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("faiss OK")

In [ ]:
# SAFE IMPORTS + CONFIG (auto-fixes the NumPy/Transformers mismatch)

import sys, subprocess, importlib

def ensure_ok():
    """
    Try to import the stack. If it fails (e.g., NumPy '_center' error),
    install compatible versions and retry.
    """
    try:
        import numpy as _np
        from sentence_transformers import SentenceTransformer as _ST
        import faiss  # noqa
    except Exception as e:
        print("Fixing Python packages…\n", e)
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-U", "--no-cache-dir",
             "numpy>=2.1,<2.3",
             "transformers>=4.44,<5",
             "sentence-transformers>=3.0.0",
             "accelerate",
             "faiss-cpu",
             "pandas"]
        )
        importlib.invalidate_caches()

ensure_ok()

# ---- now do real imports ----
from pathlib import Path
import pandas as pd, numpy as np, faiss
from sentence_transformers import SentenceTransformer

# Inputs/outputs
CSV_IN   = "legal_corpus_combined_cleaned.csv"   # produced by cleaning/tagging step
DATA_DIR = Path("data"); DATA_DIR.mkdir(parents=True, exist_ok=True)
DB_CSV   = DATA_DIR / "case_text_db.csv"         # normalized copy for search
IDX_PATH = DATA_DIR / "case_index.faiss"         # FAISS index on disk

# Retrieval model (pick one)
EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"   # higher quality (slower)
# EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # faster (good enough)


In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, faiss
from sentence_transformers import SentenceTransformer

# Inputs/outputs
CSV_IN    = "legal_corpus_combined_cleaned.csv"   # produced by cleaning step
DATA_DIR  = Path("data"); DATA_DIR.mkdir(parents=True, exist_ok=True)
DB_CSV    = DATA_DIR / "case_text_db.csv"         # normalized copy for search
IDX_PATH  = DATA_DIR / "case_index.faiss"         # FAISS index on disk

# Retrieval model (good quality)
EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"

# NOW DATASET SCRAPING COMPLETE BUILD MODEL

##Preprocessing for LLM ingestion

In [ ]:
#Preprocessing for LLM ingestion
%pip install -U faiss-cpu

In [ ]:
%pip install -U streamlit

In [ ]:
import streamlit as st
import requests

this takes such a long time...

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, faiss
from sentence_transformers import SentenceTransformer

CSV_IN   = "legal_corpus_combined_cleaned.csv"  # expects Title, Text, (optional) Source
IDX_PATH = Path("case_index.faiss")
DB_CSV   = Path("case_text_db.csv")

# 1) Load or build
if IDX_PATH.exists() and DB_CSV.exists():
    print("Loading cached FAISS index and DB…")
    index = faiss.read_index(str(IDX_PATH))
    case_df = pd.read_csv(DB_CSV)
else:
    print("Building index (first time only)…")
    case_df = pd.read_csv(CSV_IN, dtype=str).dropna(subset=["Title","Text"])
    case_df = case_df[case_df["Text"].str.strip().str.len() > 0].reset_index(drop=True)

    # 2) Encoder — small tweaks for speed
    model = SentenceTransformer("all-MiniLM-L6-v2")     # small & fast
    model.max_seq_length = 384                          # cap tokens (speedup)

    # 3) Embed (batched) and normalize for cosine
    embs = model.encode(
        case_df["Text"].tolist(),
        batch_size=64,                # try 32 if run out of RAM/CPU
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,    # cosine ready
    ).astype("float32")

    # 4) FAISS index (cosine via inner product on normalized vectors)
    dim = embs.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embs)

    # 5) Persist for instant reuse next runs
    faiss.write_index(index, str(IDX_PATH))
    case_df.to_csv(DB_CSV, index=False)
    print("Saved", IDX_PATH, "and", DB_CSV)

print("Index size:", index.ntotal)


In [ ]:
# Recreate the same encoder *only for queries*.
# Tip: keep this in memory (don’t recreate per request).
q_model = SentenceTransformer("all-MiniLM-L6-v2")
q_model.max_seq_length = 384

def search(query, k=5):
    qv = q_model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    D, I = index.search(qv, k)
    out = case_df.iloc[I[0]].copy()
    out["score"] = D[0]
    return out[["Title","score","Text"]]

# quick test
search("Visitor injured by wet floor in a supermarket — occupier liable?", k=5)


In [ ]:
import pandas as pd

# Load tagged corpus
df = pd.read_csv("legal_corpus_domain_tagged.csv", dtype=str)

# --- Make sure the columns need exist and are named consistently ---
# Some files use "Domain" instead of "LegalDomain"
if "LegalDomain" not in df.columns and "Domain" in df.columns:
    df["LegalDomain"] = df["Domain"]

# Create optional columns if missing
for c in ["Source", "Dataset"]:
    if c not in df.columns:
        df[c] = ""

# Basic required columns check
for c in ["Title", "Text", "LegalDomain"]:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

# --- Word-based chunker with small overlap (reads better than raw character chunks) ---
def chunk_row(row, chunk_words=180, overlap_words=30):
    """
    Split row['Text'] into ~chunk_words tokens with overlap to keep context.
    Keeps Title, LegalDomain, Source, Dataset for each chunk.
    """
    words = str(row["Text"]).split()
    if not words:
        return []

    chunks = []
    start = 0
    n = len(words)
    while start < n:
        end = min(start + chunk_words, n)
        part = " ".join(words[start:end])
        chunks.append({
            "Title":       row["Title"],
            "Text":        part,
            "LegalDomain": row["LegalDomain"],
            "Source":      row["Source"],
            "Dataset":     row["Dataset"],
        })
        if end >= n:
            break
        start = max(0, end - overlap_words)  # step forward with overlap

    return chunks

# --- Build the chunked table ---
chunked_records = []
for _, row in df.iterrows():
    chunked_records.extend(chunk_row(row))

df_chunks = pd.DataFrame(chunked_records)

# Save
df_chunks.to_csv("legal_corpus_chunks.csv", index=False)
print("Text chunking complete:", df_chunks.shape)


### Vector Embeddings & Retrieval (RAG)
Using embedding models (like OpenAI, Cohere, or sentence-transformers) to convert text chunks into vectors and build a retriever.

Toolkits: LangChain, LlamaIndex, or Haystack

Storage: FAISS

In [ ]:
!pip install faiss-cpu

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(df_chunks["Text"].tolist(), show_progress_bar=True)
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))

# Save index
faiss.write_index(index, "legal_faiss_index.index")


These batches load so SLOWLY

### Retrying above section as example query would not work

In [ ]:
#Retry as example above may not always work
from pathlib import Path
import pandas as pd, numpy as np, faiss
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# Assume already possess df with columns: Title, Text, LegalDomain, Source, Dataset
# If not, load it here:
# df = pd.read_csv("legal_corpus_domain_tagged.csv", dtype=str).fillna("")

# 1) Faster paragraph-aware chunker (word-based, with overlap)
def paragraph_chunk(text: str, max_words=512, overlap=64):
    """
    Make ~max_words chunks using paragraphs when possible, falling back to word-splits
    for very long paragraphs. Avoids repeated .split() on the whole string.
    """
    paras = [p.strip() for p in str(text).split("\n\n") if p.strip()]
    chunks, cur_words = [], []
    cur_len = 0

    for p in paras:
        words = p.split()
        wlen = len(words)
        # fits in current chunk
        if cur_len + wlen <= max_words:
            cur_words.extend(words)
            cur_len += wlen
        else:
            # flush current chunk if it has content
            if cur_words:
                chunks.append(" ".join(cur_words))
                # keep overlap tail
                cur_words = cur_words[-overlap:]
                cur_len = len(cur_words)

            # if the paragraph itself is too long, split it
            if wlen > max_words:
                for i in range(0, wlen, max_words - overlap):
                    seg = words[i:i + (max_words - overlap)]
                    if not seg:
                        break
                    # stitch overlap with previous tail
                    joined = " ".join((cur_words + seg) if cur_words else seg)
                    chunks.append(joined)
                    cur_words = seg[-overlap:]
                    cur_len = len(cur_words)
            else:
                # start new chunk with this paragraph
                cur_words.extend(words)
                cur_len += wlen

    if cur_words:
        chunks.append(" ".join(cur_words))

    return chunks

# 2) Build chunks + metadata efficiently
chunks, meta = [], []
for r in tqdm(df.itertuples(index=False), total=len(df)):
    cs = paragraph_chunk(r.Text, max_words=512, overlap=64)
    for c in cs:
        chunks.append(c)
        meta.append({
            "Title": getattr(r, "Title", ""),
            "Source": getattr(r, "Source", ""),
            "Dataset": getattr(r, "Dataset", ""),
            "LegalDomain": getattr(r, "LegalDomain", "")
        })

print(f"Chunked into {len(chunks):,} segments.")

# 3) Encode with batching + (GPU if available) + normalisation
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Use MiniLM for speed; for quality switch to all-mpnet-base-v2
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
enc = SentenceTransformer(MODEL_NAME, device=device)

emb = enc.encode(
    chunks,
    batch_size=128,                # bigger batch = faster on GPU; reduce if OOM
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,     # IMPORTANT if cosine wanted later
).astype("float32")

# 4) FAISS index (cosine via Inner Product on normalized vectors)
index = faiss.IndexFlatIP(dim)     # NOT L2; IP + normalized vectors == cosine
index.add(emb)
print("FAISS indexed:", index.ntotal)

# 5) (Optional) Cache to disk so you don’t redo this next run
out = Path("data"); out.mkdir(exist_ok=True)
faiss.write_index(index, str(out / "case_index.faiss"))
np.save(out / "case_embeddings.npy", emb)
pd.DataFrame(meta).assign(Text=chunks).to_csv(out / "case_text_db.csv", index=False)
print("Saved index & DB to 'data/'.")


## Query Pipeline: Retrieval + Answer Generation
On user input:

Embed the question

Retrieve top-N chunks using FAISS

Concatenate retrieved text

Generate answer using GPT or LLaMA-style model



###Step 1: Define a Query Function
This function:

Embeds the user’s legal question

Searches for top-matching chunks

Returns the most relevant text passages

In [ ]:
#Step 1: Define a query function

def search_legal_corpus(query, model, index, df_chunks, top_k=5):
    query_embedding = model.encode([query])
    D, I = index.search(np.array(query_embedding), top_k)  # D = distances, I = indices

    results = []
    for idx in I[0]:
        row = df_chunks.iloc[idx]
        results.append({
            "Title": row["Title"],
            "Dataset": row["Dataset"],
            "Legal Domain": row["Legal Domain"],
            "Source": row["Source"],
            "Text": row["Text"]
        })
    return results

###Step 2: Try Querying FAISS + Display with Legal Domain

FIRST: Rebuild FAISS Index

In [ ]:
#Step 2; Try querying FAISS + Display with legal domain
# First rebuild FAISS index
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from tqdm import tqdm
from pathlib import Path

IN_CSV   = "legal_corpus_domain_tagged.csv"
CHUNKS_CSV = "indexed_chunks_with_domain.csv"
INDEX_OUT = "faiss_legal_index.index"

# --- 1) Load safely and ensure columns exist ---
df = pd.read_csv(IN_CSV, dtype=str, encoding_errors="replace", low_memory=False)

# Required columns
for col in ["Title", "Text"]:
    if col not in df.columns:
        raise ValueError(f"{IN_CSV} is missing required column: {col}")

# Optional columns with defaults
if "Dataset" not in df.columns:     df["Dataset"] = df.get("Type", "")
if "Source" not in df.columns:      df["Source"]  = ""
if "LegalDomain" not in df.columns: df["LegalDomain"] = ""

# Drop obvious junk/empties
BAD = {"", "no title found", "no text extracted", "nan", "none", "null"}
def ok(s):
    s = str(s).strip().lower()
    return s not in BAD

df = df[df["Title"].map(ok) & df["Text"].map(ok)].reset_index(drop=True)

print("Loaded rows:", len(df))

# --- 2) Chunking ---
def split_into_chunks(text: str, max_words=200):
    if not isinstance(text, str):
        return []
    words = text.split()
    if not words:
        return []
    return [" ".join(words[i:i+max_words]) for i in range(0, len(words), max_words)]

rows = []
for row in tqdm(df.itertuples(index=False), total=len(df), desc="Chunking"):
    chunks = split_into_chunks(row.Text, max_words=200)
    for ch in chunks:
        if ch.strip():
            rows.append({
                "Title": row.Title,
                "Chunk": ch,
                "Source": row.Source,
                "Dataset": getattr(row, "Dataset", ""),
                "LegalDomain": getattr(row, "LegalDomain", "")
            })

if not rows:
    raise RuntimeError("No chunks were produced. Check that your Text column has content.")

df_chunks = pd.DataFrame(rows)
df_chunks.to_csv(CHUNKS_CSV, index=False)
print(f"Chunked rows: {len(df_chunks)} → saved to {CHUNKS_CSV}")

# --- 3) Embeddings (batched + normalized for cosine) ---
model = SentenceTransformer("all-MiniLM-L6-v2")
model.max_seq_length = 384  # speed/quality trade-off

embeddings = model.encode(
    df_chunks["Chunk"].tolist(),
    batch_size=64,                 # reduce to 32 if RAM is low
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True      # IMPORTANT: use cosine with IP index
).astype("float32")

# --- 4) FAISS index (cosine via inner product on normalized vectors) ---
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
faiss.write_index(index, INDEX_OUT)

print(f"FAISS index built with {index.ntotal} vectors → saved to {INDEX_OUT}")


In [ ]:
# Load index + chunks and run a search
idx = faiss.read_index(INDEX_OUT)
q_model = SentenceTransformer("all-MiniLM-L6-v2"); q_model.max_seq_length = 384

def search(query, k=5):
    qv = q_model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    D, I = idx.search(qv, k)
    res = df_chunks.iloc[I[0]].copy()
    res["score"] = D[0]
    return res[["Title", "LegalDomain", "Source", "score", "Chunk"]]

search("Visitor injured by wet supermarket floor — occupier liability?", 5)

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

# Load vector store and metadata
index = faiss.read_index("faiss_legal_index.index")
df_chunks = pd.read_csv("indexed_chunks_with_domain.csv")  # Same as saved earlier
meta = df_chunks.to_dict(orient="records")

# Load same model used for encoding
model = SentenceTransformer("all-MiniLM-L6-v2")

# Sample query
query = "I slipped in a shop and want to sue for negligence, what are my rights?"
query_embedding = model.encode([query])

# Retrieve top 5 relevant results
k = 3
_, indices = index.search(np.array(query_embedding).astype("float32"), k)

# Display results
for rank, i in enumerate(indices[0]):
    print(f"\nMatch #{rank + 1}")
    print("Title:", meta[i]["Title"])
    print("Legal Domain:", meta[i].get("LegalDomain", "Unknown"))
    print("Source:", meta[i].get("Source", "Unknown"))
    print("Text Snippet:\n", meta[i]["Chunk"][:500], "...\n")


In [ ]:
import faiss
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from IPython.display import display  # for Jupyter/Colab

# Load index and metadata
index = faiss.read_index("faiss_legal_index.index")
df_chunks = pd.read_csv("indexed_chunks_with_domain.csv")
meta = df_chunks.to_dict(orient="records")

# Load model
model = SentenceTransformer("all-MiniLM-L6-v2")

def search_legal_corpus(query, top_k=5):
    query_vec = model.encode([query]).astype("float32")
    distances, indices = index.search(query_vec, top_k)

    results = []
    for i, score in zip(indices[0], distances[0]):
        result = meta[i]
        result["SimilarityScore"] = float(score)
        results.append(result)

    return pd.DataFrame(results)

# Example query
query = "What are my legal rights if I was injured in a public place?"
results_df = search_legal_corpus(query, top_k=5)

# Display results
display(results_df)


### Step 3: Generate Legal Advice Answers (LLM + Retrieval-Augmented Generation)

In [ ]:
!pip install transformers

In [ ]:
from transformers import pipeline
qa_pipeline = pipeline("question-answering", model="deepset/roberta-base-squad2")

context = "You may have a claim in negligence against the supermarket. Under occupiers' liability, they owe a duty of care to visitors. If the floor was not marked or cleaned properly, they may be liable for your injuries."
question = "I slipped on wet tiles in a supermarket and injured myself. What can I do?"

result = qa_pipeline(question=question, context=context)
print(result)

In [ ]:
!pip install transformers sentencepiece

## Local Legal Advice System using flan-t5-large

### Step 1: Loading libraries and model

In [ ]:
#Load libraries and model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load FLAN-T5 Large
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")

### Step 2: Load FAISS Index and Metadata

In [ ]:
#Step 2: Load FAISS Index and Metadata
import faiss
import pandas as pd

# Load FAISS index and metadata
index = faiss.read_index("faiss_legal_index.index")
df_chunks = pd.read_csv("indexed_chunks_with_domain.csv")  # Should have Title, Text, Dataset, Legal Domain
meta = df_chunks.to_dict(orient="records")

### Step 3: Create a Retrieval Function

In [ ]:
#Step 3: Create a Retrieval Function
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def retrieve_context(query, top_k=5):
    query_vec = embedder.encode([query])
    _, indices = index.search(np.array(query_vec).astype("float32"), top_k)

    chunks = []
    for i in indices[0]:
        row = meta[i]
        context = f"Title: {row['Title']}\nDomain: {row['Legal Domain']}\nSource: {row['Dataset']}\n\n{row['Text'][:1000]}..."
        chunks.append(context)

    return "\n\n---\n\n".join(chunks)

### Step 4. Generate Advice with FLAN-T5

In [ ]:
#Step 4: Generate Advice with FLAN-T5
def generate_advice_with_flan(query):
    context = retrieve_context(query)
    prompt = (
        f"You are a legal assistant. Based on the legal materials below, provide free legal advice.\n\n"
        f"User scenario: {query}\n\n"
        f"Relevant law:\n{context}\n\n"
        f"Advice:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=300)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
def retrieve_context(query, top_k=5):
    query_vec = embedder.encode([query])
    _, indices = index.search(np.array(query_vec).astype("float32"), top_k)

    chunks = []
    for i in indices[0]:
        row = meta[i]
        title = row.get('Title', 'Unknown')
        text = row.get('Text', '')[:1000]
        domain = row.get('Legal Domain', 'Unknown')
        source = row.get('Dataset', 'Unknown')

        context = f"Title: {title}\nDomain: {domain}\nSource: {source}\n\n{text}..."
        chunks.append(context)

    return "\n\n---\n\n".join(chunks)


In [ ]:
def generate_advice_with_flan(query, top_k=5):
    context = retrieve_context(query, top_k=top_k)

    prompt = f"""You are a helpful legal advisor.
Your task is to give practical legal advice to a layperson based on UK law, citing relevant statutes or case law.

Query:
{query}

Relevant legal documents:
{context}

Advice:"""

    inputs = tokenizer(prompt, return_tensors="pt", max_length=2048, truncation=True)
    outputs = model.generate(**inputs, max_length=512)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

### Example Query

In [ ]:
query = "I slipped on wet tiles in a supermarket and injured myself. What can I do?"
response = generate_advice_with_flan(query)
print(response)

Strengthen the Prompt Further:

Updating prompt to include clearer instructions and reinforce citing legal context. Try modifying the generate_advice_with_flan function

In [ ]:
def generate_advice_with_flan(query, top_k=5):
    context = retrieve_context(query, top_k=top_k)

    prompt = f"""You are a UK legal expert. A person is asking for legal advice based on their situation.
Use the relevant legal documents provided to answer. Be specific and mention any related statutes or case law.

Person's question:
{query}

Relevant legal documents:
{context}

Give your legal advice clearly and concisely:"""

    inputs = tokenizer(prompt, return_tensors="pt", max_length=2048, truncation=True)
    outputs = model.generate(**inputs, max_length=512, num_beams=4, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


Filter Context to Only Tort Law
Query is clearly tort-based (slipping in a supermarket). If the model is being confused by unrelated statute or contract law input, ability filter the FAISS context to include only "Tort" domain entries.

Update retrieve_context()

In [ ]:
#Update retrieve_context()
def retrieve_context(query, top_k=5, domain_filter="Tort"):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, top_k)

    chunks = []
    for i in indices[0]:
        row = meta[i]
        if domain_filter and row.get("Legal Domain", "") != domain_filter:
            continue
        context = f"Title: {row['Title']}\nDomain: {row['Legal Domain']}\nSource: {row['Dataset']}\n\n{row['Text'][:1000]}..."
        chunks.append(context)

    return "\n\n---\n\n".join(chunks[:top_k])

## Corrected setup

In [ ]:
#Library import again for ease
from sentence_transformers import SentenceTransformer
from transformers import T5Tokenizer, T5ForConditionalGeneration
import faiss
import pandas as pd
import numpy as np

In [ ]:
# For embedding / retrieval
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# For legal text generation
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large")

#### Updated Retrieval Function (with embedding model)

In [ ]:
#Updated Retrieval Function (with embedding model)
def retrieve_context(query, top_k=5, domain_filter="Tort"):
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(np.array(query_embedding), top_k)

    chunks = []
    for i in indices[0]:
        row = meta[i]
        if domain_filter and row.get("Legal Domain", "") != domain_filter:
            continue
        context = f"Title: {row['Title']}\nDomain: {row['Legal Domain']}\nSource: {row['Dataset']}\n\n{row['Text'][:1000]}..."
        chunks.append(context)

    return "\n\n---\n\n".join(chunks[:top_k])


#### Generation Function Using flan-t5

In [ ]:
#Generation Function using FLAN-T5
def generate_advice_with_flan(query, top_k=5):
    context = retrieve_context(query, top_k)

    prompt = f"""You are a UK legal expert. A person is asking for legal advice based on their situation.
Use the relevant legal documents provided to answer. Be specific and mention any related statutes or case law.

Person's question:
{query}

Relevant legal documents:
{context}

Give your legal advice clearly and concisely:"""

    inputs = tokenizer(prompt, return_tensors="pt", max_length=2048, truncation=True)
    outputs = model.generate(**inputs, max_length=512, num_beams=4, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

#### Example Query 2

In [ ]:
query = "I slipped on wet tiles in a supermarket and injured myself. What can I do?"
response = generate_advice_with_flan(query)
print(response)

## Tweaking as example query did not make sense
1. Refine the Prompt to Force Legal Reasoning

In [ ]:
#Refine the prompt to force legal reasoning
prompt = f"""You are a UK legal expert assisting someone who cannot afford a lawyer.
They are asking for legal advice based on their real-life situation.

Refer ONLY to the provided legal text — cite statutes and case law.

Question:
{query}

Relevant legal materials:
{context}

Please explain what legal rights the person has, and what action they can take."""

Use context[:500] per chunk instead of [:1000]
flan-t5-large is easily overwhelmed by long text. Try retrieving shorter, more focused chunks. For instance

Updated retrieve_context() Function

In [ ]:
#Updated retrieve_context() Function
def retrieve_context(query, top_k=5):
    query_embedding = model.encode([query])
    _, indices = index.search(np.array(query_embedding), top_k)

    chunks = []
    for i in indices[0]:
        row = meta[i]
        title = row.get("Title", "Unknown")
        domain = row.get("Legal Domain", "Unknown")
        source = row.get("Dataset", "Unknown")
        text = row.get("Text", "")[:500]

        context = f"Title: {title}\nDomain: {domain}\nSource: {source}\n\n{text}..."
        chunks.append(context)

    return "\n---\n".join(chunks)

### Fixing syntax error

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load FLAN-T5 model and tokenizer
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large")

def retrieve_context(query, top_k=5):
    # Dummy version – replace with actual vector search
    return [
        "Title: Occupiers’ Liability Act 1957\nDomain: Tort\nSource: Legislation.gov.uk\n\nThis Act outlines the duty of care owed by those who control premises to visitors...",
        "Title: Donoghue v Stevenson\nDomain: Tort\nSource: BAILII.org\n\nThe landmark case establishing the modern law of negligence, imposing a duty of care on manufacturers to end users...",
        "Title: Tomlinson v Congleton\nDomain: Tort\nSource: BAILII.org\n\nA decision clarifying that occupiers are not liable when individuals engage in obvious risky behaviour..."
    ][:top_k]

def generate_advice_with_flan(query, top_k=5):
    # Retrieve top-k legal chunks
    context_chunks = retrieve_context(query, top_k)

    # Build the prompt
    prompt = f"Legal question: {query}\n\nRelevant legal context:\n"
    for chunk in context_chunks:
        prompt += f"{chunk}\n\n"
    prompt += "Give legal advice in simple terms."

    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    # Generate output
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=256,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

#### Example Query 4

In [ ]:
query = "My landlord hasn't repaired a broken boiler for two weeks during winter. What are my legal rights?"
response = generate_advice_with_flan(query)
print(response)

### Improve Output Structure (Citations + Clarity)

Update generate_advice_with_flan() so advice is followed by citations from the retrieved context.

In [ ]:
#Update generate_advice_with_flan() so advice is followed by citations from the retrieved context
def generate_advice_with_flan(query, top_k=5):
    context_chunks = retrieve_context(query, top_k)

    prompt = f"You are a legal assistant. A user asked:\n\"{query}\"\n\nRelevant legal materials:\n"
    for i, chunk in enumerate(context_chunks):
        prompt += f"[{i+1}] {chunk.strip()}\n\n"
    prompt += "Based on the materials above, give clear legal advice in simple terms. Refer to the sources by number if helpful."

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=256,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

### FINAL STAGE: Web UI (Useful for ease of user access)
Build a simple interactive front-end:

In Jupyter/Colab: Use gradio or streamlit

Locally: Flask + HTML form

Hosted: Hugging Face Spaces (with gradio)

In [ ]:
pip install gradio

In [ ]:
import gradio as gr

iface = gr.Interface(fn=generate_advice_with_flan,
                     inputs="text",
                     outputs="text",
                     title="Free Legal Advice Assistant",
                     description="Ask a legal question (UK law, tort or contract).")

iface.launch()

Notes: this is working but the runtime is very slow, the text sent back is also littered with ellipses, returned results are often incorrect

### Fix Strategy

####Step 1: Improve Semantic Search Quality
Use better chunking (paragraph- or sentence-level, not fixed-length), and make sure embeddings are meaningful.


Increase top_k from 3 to 8.

Filter by legal domain more strictly.

Boost relevance by reranking (optional but powerful).

Update retrieve_context() like this

In [ ]:
#Step 1: Improve semantic search quality
def retrieve_context(query, top_k=8):
    query_vec = model.encode([query])
    _, indices = index.search(np.array(query_vec).astype('float32'), top_k)

    chunks = []
    for i in indices[0]:
        row = meta[i]
        # Filter out unrelated domains if needed
        if row.get("Legal Domain", "").lower() not in ["tort", "personal injury"]:
            continue
        context = f"Title: {row['Title']}\nDomain: {row['Legal Domain']}\nSource: {row['Dataset']}\n\n{row['Text'][:1000]}..."
        chunks.append(context)

    return "\n\n".join(chunks[:top_k])


#### Step 2: Prompt Template Fix
The generator may not be instructed clearly. Add clarity here

In [ ]:
#Step 2: Prompt templatfix
prompt = f"""You are a helpful legal assistant providing UK legal advice based on retrieved laws and cases.

Here is the user's situation:
"{query}"

Here are the most relevant legal references:
{context}

Please provide a clear, practical answer for the user, and mention any applicable legal principles or cases."""

#### Step 3: Improve Retrieval Filtering (by Legal Domain)

In [ ]:
#Step 3: Improve Retrieval Filtering (by Legal Domain)
from sentence_transformers import SentenceTransformer

# This is for embeddings, not text generation
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def retrieve_context(query, top_k=8, filter_domain=None):
    query_emb = embedder.encode([query])  # use SentenceTransformer here
    _, indices = index.search(query_emb, top_k * 3)

    chunks = []
    count = 0

    for i in indices[0]:
        row = meta[i]
        domain = row.get("Legal Domain", "").strip().lower()

        if filter_domain and filter_domain.lower() != domain:
            continue

        context = (
            f"Title: {row['Title']}\n"
            f"Domain: {row.get('Legal Domain', 'Unknown')}\n"
            f"Source: {row['Dataset']}\n\n"
            f"{row['Text'][:1000]}..."
        )
        chunks.append(context)
        count += 1
        if count >= top_k:
            break

    return "\n---\n".join(chunks)

#### Step 4: Improved prompt for legal advice

In [ ]:
#Step 4: Improved prompt for legal advice
def generate_advice_with_flan(query, domain_hint="Tort"):
    context = retrieve_context(query, top_k=8, filter_domain=domain_hint)

    prompt = (
        f"You are a UK legal advisor helping self-representing individuals.\n"
        f"Use only the provided legal materials to answer.\n"
        f"If unsure, say so clearly.\n"
        f"User's situation:\n{query}\n\n"
        f"Relevant legal materials:\n{context}\n\n"
        f"Advice:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda" if torch.cuda.is_available() else "cpu")
    outputs = model.generate(**inputs, max_new_tokens=300)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

## Improving accruacy

### 3 Main Paths to Improve Accuracy;

#### Step 1. Fine-tune a Language Model (most effective but resource-heavy)
Train a model like flan-t5 on  legal dataset so it better understands how to generate legal advice using UK-specific legal corpus.

How:
Format dataset as instruction → response pairs (like: scenario → legal advice)

Use Hugging Face's Trainer API or LoRA-based adapters to fine-tune flan-t5 (or smaller models as have limited resources)

Start with ~1,000 examples for a proof of concept

In [ ]:
pip install transformers datasets peft accelerate

#### Step 2. Use RAG + Prompt Tuning (great balance of cost and performance)
Keep the model frozen but improve prompts and retrieval quality.

Tips:
Ensure that each chunk in FAISS index is concise, relevant, and labeled by domain

Use domain-filtered retrieval (domain="Tort" or Contract) to cut noise

Format prompt:

In [ ]:
#Step 2: Use RAG + Prompt tuning
def build_prompt(user_query, retrieved, max_items=5, max_chars=800):
    """Build a safe prompt from a user query and retrieved context (string, list of dicts, or DataFrame)."""
    # format context robustly
    def format_context(r):
        # if it's already a string
        if isinstance(r, str):
            return r.strip()

        # if it's a DataFrame, turn into records
        try:
            import pandas as pd
            if isinstance(r, pd.DataFrame):
                r = r.to_dict("records")
        except Exception:
            pass

        parts = []
        for item in list(r)[:max_items]:
            # dict-like or object-like access
            get = item.get if hasattr(item, "get") else lambda k, d="": getattr(item, k, d)
            title = get("Title") or "(Untitled)"
            text  = get("Text") or get("Chunk") or ""
            source = get("Source") or ""
            snippet = (text[:max_chars] + "...") if len(text) > max_chars else text
            parts.append(f"Title: {title}\nSource: {source}\nExcerpt:\n{snippet}")
        return "\n\n---\n\n".join(parts) if parts else "(no context found)"

    ctx = format_context(retrieved)
    user_query = user_query or ""

    return (
        "You are a legal advisor.\n\n"
        "Given the scenario:\n"
        f"\"{user_query}\"\n\n"
        "Here is relevant law and case context:\n"
        f"{ctx}\n\n"
        "What legal advice would you give in plain English?"
    )

# -------- EXAMPLE (works as-is) --------
user_query = "I slipped on a wet floor in a supermarket and broke my wrist."
retrieved = [
    {
        "Title": "Occupiers' Liability Act 1957",
        "Text": "Sets duty of care owed to lawful visitors, including taking reasonable care for their safety.",
        "Source": "Legislation.gov.uk"
    },
    {
        "Title": "Tomlinson v Congleton Borough Council",
        "Text": "Limits liability where a claimant willingly accepts an obvious risk; context matters.",
        "Source": "BAILII"
    }
]

PROMPT = build_prompt(user_query, retrieved)
print(PROMPT)

### Step 3. Synthetic Training Data
Run many example queries through RAG pipeline

Curate the top outputs

Use these QA pairs to fine-tune the model later (bootstrapping)

a. Implementation Steps (For RAG + Light Fine-tuning)

Create JSONL dataset like:

In [ ]:
#Step 3: Synthetic training data, run many example queries through RAG pipeline, curate top outputs, use QA pairs to fine tune the model later
#Implementation setps:
#Create JSONL dataset:
{"instruction": "My landlord hasn’t fixed my boiler for weeks. What are my rights?", "context": "Housing Act 1988...\nCase: Smith v Jones...", "response": "You may have a claim under Section X..."}


b. Train using Hugging Face’s Trainer or LoRA:a.

In [ ]:
# Recommended in Colab because it installs into the current kernel:
#Train using Hugging Face's Trainer or LoRA;a
%pip install -q peft transformers datasets accelerate


c. Start with flan-t5-small for fast iterations:

In [ ]:
#Start with FLAN-T5-small for fast iterations
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Trainer, TrainingArguments

model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")

## 4–6) Retrieval + Generator + Smoke test (added)

In [ ]:
# 4. Retrieval — resilient loader + cosine search (+ optional rerank)
from pathlib import Path
import pandas as pd, numpy as np
from sentence_transformers import SentenceTransformer

# ---- Config (uses existing names if already defined) ----
DATA_DIR  = Path(globals().get("DATA_DIR", "data"))
META_PATH = Path(globals().get("META_PATH", DATA_DIR / "legal_chunks.parquet"))
EMB_PATH  = Path(globals().get("EMB_PATH",  DATA_DIR / "legal_embeddings.npy"))
EMBED_MODEL = globals().get("EMBED_MODEL", "sentence-transformers/all-mpnet-base-v2")
TOP_K = int(globals().get("TOP_K", 6))
USE_RERANKER = bool(globals().get("USE_RERANKER", True))
CROSSENCODER_MODEL = globals().get("CROSSENCODER_MODEL", "cross-encoder/ms-marco-MiniLM-L-6-v2")

DATA_DIR.mkdir(parents=True, exist_ok=True)

def _standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    # lower-case column names for consistency
    df = df.rename(columns={c: c.lower() for c in df.columns})
    # prefer 'text' column; fall back to 'chunk'
    if "text" not in df.columns and "chunk" in df.columns:
        df["text"] = df["chunk"]
    # prefer 'domain'; fall back to 'legaldomain'
    if "domain" not in df.columns and "legaldomain" in df.columns:
        df["domain"] = df["legaldomain"]
    # prefer 'uri'; fall back to 'source'
    if "uri" not in df.columns and "source" in df.columns:
        df["uri"] = df["source"]
    # ensure required columns exist
    for col, default in [("title","(Untitled)"), ("text",""), ("domain","Unknown"), ("uri","")]:
        if col not in df.columns:
            df[col] = default
    # basic cleaning
    df = df[(df["text"].astype(str).str.strip()!="") & (df["title"].astype(str).str.strip()!="")].reset_index(drop=True)
    return df[["title","text","domain","uri"]]

def _load_chunks_and_embeddings():
    # 1) Try the intended artifacts
    if META_PATH.exists() and EMB_PATH.exists():
        try:
            chunks_df = pd.read_parquet(META_PATH)
        except Exception:
            # Parquet engine missing? try CSV mirror
            csv_alt = META_PATH.with_suffix(".csv")
            if not csv_alt.exists():
                raise
            chunks_df = pd.read_csv(csv_alt, dtype=str, encoding_errors="replace", low_memory=False)
        emb = np.load(EMB_PATH)
        return _standardize_columns(chunks_df), emb

    # 2) Fallback: build from CSV already produced in earlier steps
    for candidate in [Path("indexed_chunks_with_domain.csv"), Path("legal_corpus_chunks.csv")]:
        if candidate.exists():
            df = pd.read_csv(candidate, dtype=str, encoding_errors="replace", low_memory=False)
            df = _standardize_columns(df)
            # build embeddings now
            enc = SentenceTransformer(EMBED_MODEL)
            enc.max_seq_length = 384
            emb = enc.encode(
                df["text"].tolist(),
                batch_size=64, convert_to_numpy=True, normalize_embeddings=True
            ).astype("float32")
            # persist for next time (Parquet if possible, else CSV)
            try:
                df.to_parquet(META_PATH, index=False)
            except Exception:
                df.to_csv(META_PATH.with_suffix(".csv"), index=False)
            np.save(EMB_PATH, emb)
            print(f"Built and saved embeddings: {EMB_PATH}")
            return df, emb

    # 3) As a last resort, tell the user what to run
    raise FileNotFoundError(
        f"Missing {META_PATH} and {EMB_PATH}. "
        "Run your chunking/build step to create either 'indexed_chunks_with_domain.csv' "
        "or 'legal_corpus_chunks.csv' first."
    )

# ---- Load (or build) corpus + embeddings ----
chunks_df, emb = _load_chunks_and_embeddings()

# ---- Query encoder ----
query_encoder = SentenceTransformer(EMBED_MODEL)
query_encoder.max_seq_length = 384

# ---- Optional reranker (CrossEncoder) ----
reranker = None
if USE_RERANKER:
    try:
        from sentence_transformers import CrossEncoder
        reranker = CrossEncoder(CROSSENCODER_MODEL)
        print("Reranker loaded:", CROSSENCODER_MODEL)
    except Exception as e:
        print("Reranker not available:", e)

def retrieve_context(query: str, top_k: int = TOP_K, domain: str | None = None, as_strings: bool = False):
    # cosine similarities (embeddings are normalized)
    qv = query_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0].astype("float32")
    sims = emb @ qv

    # candidate pool, then sort
    pool = max(top_k * 8, top_k)
    idx = np.argpartition(-sims, pool)[:pool]
    order = idx[np.argsort(-sims[idx])]
    cand = chunks_df.iloc[order].copy()
    cand["score"] = sims[order]

    # optional domain filter if enough results remain
    if domain:
        sub = cand[cand["domain"].str.lower() == domain.lower()]
        if len(sub) >= max(3, top_k // 2):
            cand = sub

    # optional rerank
    if reranker is not None:
        pairs = [(query, t) for t in cand["text"].tolist()]
        rr = reranker.predict(pairs)
        cand = cand.assign(rerank=rr).sort_values("rerank", ascending=False)
    else:
        cand = cand.sort_values("score", ascending=False)

    cand = cand.head(top_k)

    if as_strings:
        return [
            f"Title: {r.title}\nDomain: {getattr(r,'domain','Unknown')}\nSource: {getattr(r,'uri','')}\n\n{r.text}"
            for r in cand.itertuples()
        ]
    return cand.to_dict(orient="records")


In [ ]:
# 5.Generator — FLAN-T5-Large (efficient + accurate)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch, re

tok = AutoTokenizer.from_pretrained(GEN_MODEL, use_fast=True)
dtype = torch.bfloat16 if torch.cuda.is_available() else None

gen = AutoModelForSeq2SeqLM.from_pretrained(
    GEN_MODEL,
    torch_dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True
).eval()

try:
    gen = torch.compile(gen)
except Exception:
    pass

GEN_KW = dict(
    max_new_tokens=256,
    do_sample=False,
    temperature=0.0,
    num_beams=1,
    no_repeat_ngram_size=3,
    repetition_penalty=1.05,
)

def pack_context(passages, max_chars=3000):
    best_by_uri = {}
    for p in passages:
        u = p["uri"]
        if u not in best_by_uri or p["score"] > best_by_uri[u]["score"]:
            best_by_uri[u] = p
    dedup = sorted(best_by_uri.values(), key=lambda x: x["score"], reverse=True)
    ctx_lines, used = [], 0
    for i, p in enumerate(dedup, start=1):
        header = f"[{i}] {p['title']} — {p['section']} | {p['uri']}"
        block = f"{header}\n{p['text']}\n"
        if used + len(block) > max_chars: break
        ctx_lines.append(block); used += len(block)
    return "\n".join(ctx_lines)

PROMPT = """You are a cautious legal explainer for UK law.
Use ONLY the evidence below. Quote sparingly and always add inline citations like [1], [2].
If evidence is insufficient, say so and ask for a narrower question.

Question: {q}

Evidence:
{ctx}

Write:
1) Short answer (2–4 sentences) with inline [n] cites.
2) Why these sources matter (bullet points), each bullet ends with a [n] cite.
3) References: list [n] → Title — Section — URI.
"""

@torch.no_grad()
def generate_advice(query: str, passages: list, *, k_used: int | None = None):
    ctx = pack_context(passages)
    prompt = PROMPT.format(q=query, ctx=ctx)
    ids = tok(prompt, return_tensors="pt", truncation=True)
    if torch.cuda.is_available():
        ids = ids.to(gen.device)
    out = gen.generate(**ids, **GEN_KW)
    answer = tok.decode(out[0], skip_special_tokens=True)
    cited = {int(n) for n in re.findall(r"\[(\d+)\]", answer)}
    cap = k_used or len(passages)
    if not cited or any(n < 1 or n > cap for n in cited):
        answer += "\n\n[Note] Citation check: please verify references."
    return answer


In [ ]:
# --- Drop-in: safe context packer + advice generator ---

def pack_context(passages, max_chars=3500):
    """Normalize passages (list/DF), dedupe, and pack into a context string."""
    import pandas as pd

    # Normalize to list of dicts
    if isinstance(passages, pd.DataFrame):
        records = passages.to_dict("records")
    else:
        records = list(passages)

    # Standardize keys & provide fallbacks
    norm = []
    for p in records:
        get = p.get if hasattr(p, "get") else lambda k, d=None: getattr(p, k, d)
        title   = get("title") or get("Title") or "(Untitled)"
        text    = get("text") or get("Text") or get("Chunk") or ""
        uri     = get("uri") or get("URL") or get("Url") or get("Source") or ""
        section = get("section") or get("Section") or get("domain") or get("LegalDomain") or ""
        norm.append({"title": title, "text": text, "uri": uri, "section": section})

    # De-duplicate (by title + first 200 chars of text)
    seen, dedup = set(), []
    for p in norm:
        key = (p["title"], p["text"][:200])
        if key in seen:
            continue
        seen.add(key)
        dedup.append(p)

    # Build blocks within max_chars
    ctx_lines, used = [], 0
    for i, p in enumerate(dedup, start=1):
        header = f"[{i}] {p['title']}"
        if p["section"]:
            header += f" — {p['section']}"
        if p["uri"]:
            header += f" | {p['uri']}"
        block = f"{header}\n{p['text']}\n"
        if used + len(block) > max_chars:
            break
        ctx_lines.append(block)
        used += len(block)

    return "\n".join(ctx_lines), min(i, len(dedup)) if dedup else 0


def build_prompt(user_query, context_str):
    return (
        "You are a legal advisor.\n\n"
        "Given the scenario:\n"
        f"\"{user_query}\"\n\n"
        "Here is relevant law and case context:\n"
        f"{context_str}\n\n"
        "What legal advice would you give in plain English?"
    )


def generate_advice(user_query, passages, k_used=None):
    # Pack context safely (no KeyError on 'section')
    context_str, used = pack_context(passages, max_chars=3500)
    prompt = build_prompt(user_query, context_str)

    # If generator model, call it here. Otherwise return the prompt for now.
    try:
        # Example with a text-generation pipeline as one set up:
        # out = text_generator(prompt, max_new_tokens=400, do_sample=False)[0]["generated_text"]
        # return out
        pass
    except Exception:
        pass

    # Fallback: return the prompt so there is ability feed it to  generator elsewhere
    return prompt


In [ ]:
q = "Visitor injured by a loose stair tile — is the occupier liable?"
P = retrieve_context(q, top_k=6, domain="Tort", as_strings=False)  # returns title/text/domain/uri
print(generate_advice(q, P, k_used=len(P))[:1200])


In [ ]:
# --- Gradio UI for Legal RAG ---
try:
    import gradio as gr
except Exception:
    # If gradio isn't installed, uncomment the next line once, run, then re-run this cell
    # %pip install gradio
    import gradio as gr

import pandas as pd
import time

DOMAINS = ["Any", "Tort", "Contract", "Employment", "Consumer", "Property"]

def _format_table(passages):
    if isinstance(passages, pd.DataFrame):
        df = passages.copy()
    else:
        df = pd.DataFrame(passages)
    # normalize columns
    rename = {}
    for a,b in [("Title","title"),("Text","text"),("Chunk","text"),
                ("Source","uri"),("URL","uri"),("Url","uri"),
                ("LegalDomain","domain")]:
        if a in df.columns and b not in df.columns:
            rename[a]=b
    df = df.rename(columns=rename)
    keep = [c for c in ["title","domain","uri","score","text"] if c in df.columns]
    if "text" in keep:
        df["text"] = df["text"].astype(str).str.slice(0,400) + "…"
    return df[keep].head(10)

def advise_fn(scenario, domain, k):
    if not scenario or not scenario.strip():
        return "Please describe your situation.", pd.DataFrame()
    start = time.time()
    dom = None if (domain == "Any") else domain
    passages = retrieve_context(scenario, top_k=int(k), domain=dom, as_strings=False)
    table = _format_table(passages)
    advice = generate_advice(scenario, passages, k_used=len(passages))
    ms = int((time.time() - start)*1000)
    header = f"**Response time:** {ms} ms • **Results used:** {len(passages)}"
    return header + "\n\n" + (advice if isinstance(advice, str) else str(advice)), table

with gr.Blocks(title="Legal Scenario Advisor") as demo:
    gr.Markdown("## Legal Scenario Advisor (RAG demo) — not legal advice")
    with gr.Row():
        scenario = gr.Textbox(lines=7, label="Describe your situation")
        with gr.Column(scale=0.5):
            domain = gr.Dropdown(choices=DOMAINS, value="Any", label="Domain filter")
            k = gr.Slider(3, 10, value=6, step=1, label="Top-k passages")
            go = gr.Button("Get Advice", variant="primary")
    advice_out = gr.Markdown(label="Advice")
    table_out  = gr.Dataframe(headers=["title","domain","uri","score","text"], label="Retrieved passages", wrap=True)
    go.click(advise_fn, inputs=[scenario, domain, k], outputs=[advice_out, table_out])

# Launch (in notebooks)
demo.launch(share=False)  # set share=True if wanting a public link (Colab)


## Evaluation metrics

expects a dev set DataFrame like:
eval_df with columns:

query (str)

relevant_uris (list[str]) or relevant_titles (list[str])
(use whichever able to label; URIs are best)

In [ ]:
import math, re, time
import pandas as pd

def _first(x):  # helper: first non-empty
    for v in x:
        if v: return v
    return ""

def _norm(s):
    return re.sub(r"\s+", " ", str(s).strip().lower())

def retrieval_eval(eval_df, k=6, use_uris=True, domain=None):
    """
    eval_df columns:
      - query: str
      - relevant_uris: list[str]   (preferred)
        or relevant_titles: list[str]  (fallback)
    Returns macro-averaged Recall@k, MRR@k, nDCG@k and per-query details.
    """
    rows = []
    Rk_total, MRR_total, nDCG_total = 0.0, 0.0, 0.0
    n = 0

    for r in eval_df.itertuples():
        q = r.query
        gold = getattr(r, "relevant_uris", None) if use_uris else getattr(r, "relevant_titles", None)
        if gold is None:
            # fall back if the column isn't there
            gold = getattr(r, "relevant_titles", None) or getattr(r, "relevant_uris", None)
        gold = set(_norm(x) for x in (gold or []) if str(x).strip())
        if not gold:
            continue

        # run retrieval
        cand = retrieve_context(q, top_k=k, domain=domain, as_strings=False)

        # make comparable strings
        got = []
        for c in cand:
            uri = _norm(c.get("uri") or c.get("URL") or c.get("Source") or "")
            title = _norm(c.get("title") or c.get("Title") or "")
            key = uri if use_uris else title
            if not key: key = uri or title
            got.append(key)

        # compute metrics
        rel_flags = [1 if g in gold else 0 for g in got]
        # Recall@k (binary relevance)
        hit_any = 1 if any(rel_flags) else 0
        Rk_total += hit_any

        # MRR@k
        rr = 0.0
        for i, flag in enumerate(rel_flags, start=1):
            if flag:
                rr = 1.0 / i
                break
        MRR_total += rr

        # nDCG@k (binary gains)
        DCG = sum(flag / math.log2(i+1) for i, flag in enumerate(rel_flags, start=1))
        ideal = [1] * min(len(gold), k)
        IDCG = sum(1.0 / math.log2(i+1) for i,_ in enumerate(ideal, start=1)) or 1.0
        nDCG_total += DCG / IDCG

        rows.append({
            "query": q,
            "hits": rel_flags,
            "RR": rr,
            "has_hit": hit_any,
            "nDCG": DCG / IDCG,
        })
        n += 1

    summ = {
        "Queries": n,
        f"Recall@{k}": round(Rk_total / max(n,1), 4),
        f"MRR@{k}": round(MRR_total / max(n,1), 4),
        f"nDCG@{k}": round(nDCG_total / max(n,1), 4),
    }
    return pd.DataFrame(rows), summ

def token_f1(pred: str, ref: str):
    from collections import Counter
    tp = 0
    p = re.findall(r"\w+", str(pred).lower())
    r = re.findall(r"\w+", str(ref).lower())

In [ ]:
# Build a tiny labeled set by hand (URIs/titles from corpus)
mini = pd.DataFrame([
    {
        "query": "Visitor slipped on wet supermarket floor; is the store liable?",
        "relevant_titles": ["occupiers' liability act 1957", "tomlinson v congleton"]
    },
    {
        "query": "Goods arrived late and breached delivery terms; remedies?",
        "relevant_titles": ["sale of goods act", "hadley v baxendale"]
    },
])

# Retrieval metrics (title matching as we didn't add URIs here)
rows, summary = retrieval_eval(mini, k=6, use_uris=False, domain=None)
summary, rows.head()


Retrieval quality: Recall@k, MRR@k, nDCG@k on dev set.

Show an A/B table: MiniLM vs mpnet, with and without the CrossEncoder reranker.

Answer quality: token-F1 vs references (if any), and Context Coverage (groundedness proxy).

Also include 5–10 human-rated samples using a simple rubric: correctness, clarity, citations to retrieved items, and cautions/disclaimers.

Systems metrics: end-to-end latency (p50/p95), memory footprint, index size, build time.

Ablations: chunk size (e.g., 150 vs 300 words), k, domain filter on/off, reranker on/off.

In [ ]:
import math, re, time, statistics
import pandas as pd

def _norm(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip().lower())

def _answer_tokens(s: str):
    return re.findall(r"\w+", str(s).lower())

def context_coverage(answer, passages) -> float:
    """% of answer tokens that appear in retrieved context."""
    if isinstance(passages, pd.DataFrame):
        ctx = " ".join(passages.get("text","").astype(str).tolist())
    else:
        ctx = " ".join([(p.get("text") or p.get("Chunk") or "") for p in passages])
    A = set(_answer_tokens(answer))
    C = set(_answer_tokens(ctx))
    return 0.0 if not A else round(len(A & C) / len(A), 4)

def token_f1(pred: str, ref: str) -> float:
    """Lexical F1 vs reference (only if you have references)."""
    from collections import Counter
    p = Counter(_answer_tokens(pred)); r = Counter(_answer_tokens(ref))
    common = sum(min(p[w], r[w]) for w in set(p)|set(r))
    precision = common / max(1, sum(p.values()))
    recall    = common / max(1, sum(r.values()))
    return 0.0 if (precision+recall)==0 else round(2*precision*recall/(precision+recall), 4)


### Weak labels from corpus (fast sanity check)

In [ ]:
# Pull distinct titles from chunks mapping
try:
    corpus = pd.read_csv("indexed_chunks_with_domain.csv")
except Exception:
    corpus = pd.read_csv("legal_corpus_domain_tagged.csv")

title_col = "Title" if "Title" in corpus.columns else "title"
domain_col = "LegalDomain" if "LegalDomain" in corpus.columns else ("domain" if "domain" in corpus.columns else None)

sample = (corpus[[title_col, domain_col]].drop_duplicates() if domain_col else corpus[[title_col]].drop_duplicates()).sample(20, random_state=42)

def title_to_query(t):
    t = re.sub(r"\[[^\]]+\]", "", str(t))
    t = re.sub(r"\s+", " ", t).strip()
    return f"What legal principle applies in {t}?"

eval_df = pd.DataFrame({
    "query": sample[title_col].apply(title_to_query),
    "relevant_titles": sample[title_col].apply(lambda x: [str(x)]),
})
eval_df.head()


### Retrieval metrics: Recall@k, MRR@k, nDCG@k

In [ ]:
def retrieval_eval(eval_df: pd.DataFrame, k=6, use_uris=False, domain=None):
    rows = []
    Rk = MRR = nDCG = 0.0
    n = 0

    for rec in eval_df.itertuples():
        q = rec.query
        gold = getattr(rec, "relevant_uris", None) if use_uris else getattr(rec, "relevant_titles", None)
        if gold is None:  # fallbacks
            gold = getattr(rec, "relevant_titles", None) or getattr(rec, "relevant_uris", None)
        gold = set(_norm(x) for x in (gold or []) if str(x).strip())
        if not gold:
            continue

        # Retrieve
        cand = retrieve_context(q, top_k=k, domain=domain, as_strings=False)

        # Keys to match against gold
        got = []
        for c in cand:
            key = _norm(c.get("uri") or "") if use_uris else _norm(c.get("title") or c.get("Title") or "")
            if not key: key = _norm(c.get("title") or c.get("Title") or c.get("uri") or "")
            got.append(key)

        rel = [1 if g in gold else 0 for g in got]

        # Recall@k
        hit = 1 if any(rel) else 0
        Rk += hit

        # MRR@k
        rr = 0.0
        for i, flag in enumerate(rel, start=1):
            if flag:
                rr = 1.0 / i; break
        MRR += rr

        # nDCG@k (binary gains)
        DCG = sum(flag / math.log2(i+1) for i, flag in enumerate(rel, start=1))
        IDCG = sum(1.0 / math.log2(i+1) for i in range(1, min(len(gold), k)+1)) or 1.0
        nDCG += DCG / IDCG

        rows.append({"query": q, "hits": rel, "RR": rr, "has_hit": hit, "nDCG": DCG/IDCG})
        n += 1

    summary = {
        "Queries": n,
        f"Recall@{k}": round(Rk / max(n,1), 4),
        f"MRR@{k}": round(MRR / max(n,1), 4),
        f"nDCG@{k}": round(nDCG / max(n,1), 4),
    }
    return pd.DataFrame(rows), summary

# Run it
r_rows, r_summary = retrieval_eval(eval_df, k=6, use_uris=False, domain=None)
print(r_summary)
r_rows.head()


### Generation metrics without references (Coverage + Safety)

In [ ]:
def generation_eval(eval_df: pd.DataFrame, k=6, domain=None):
    out = []
    latencies = []
    for rec in eval_df.itertuples():
        q = rec.query
        t0 = time.time()
        P = retrieve_context(q, top_k=k, domain=domain, as_strings=False)
        ans = generate_advice(q, P, k_used=len(P))
        t1 = time.time()

        cov = context_coverage(ans, P)
        has_disclaimer = bool(re.search(r"not (legal|professional) advice", ans.lower()))
        out.append({
            "query": q,
            "coverage": cov,
            "has_disclaimer": has_disclaimer,
            "answer_len": len(ans),
            "latency_ms": int((t1 - t0)*1000),
        })
        latencies.append((t1 - t0)*1000)

    df = pd.DataFrame(out)
    summary = {
        "coverage_mean": round(df["coverage"].mean(), 4),
        "coverage_median": round(df["coverage"].median(), 4),
        "answers_with_disclaimer_%": round(100*df["has_disclaimer"].mean(), 1),
        "answer_len_mean": int(df["answer_len"].mean()),
        "latency_p50_ms": int(statistics.median(latencies)) if latencies else 0,
        "latency_p95_ms": int(sorted(latencies)[int(0.95*len(latencies))-1]) if latencies else 0,
    }
    return df, summary

g_rows, g_summary = generation_eval(eval_df, k=6, domain=None)
print(g_summary)
g_rows.head()


In [ ]:
import math, re
import pandas as pd

def _norm(s: str) -> str:
    """Normalize keys (titles/URIs) for matching."""
    s = str(s or "").strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def retrieval_eval(eval_df: pd.DataFrame, k=6, use_uris=False, domain=None):
    rows = []
    Rk = MRR = nDCG_sum = 0.0
    n = 0

    for rec in eval_df.itertuples():
        q = rec.query
        # Gold set: prefer URIs if requested, else titles; fallback to whatever exists.
        gold_raw = getattr(rec, "relevant_uris", None) if use_uris else getattr(rec, "relevant_titles", None)
        if gold_raw is None:
            gold_raw = getattr(rec, "relevant_titles", None) or getattr(rec, "relevant_uris", None)
        gold = { _norm(x) for x in (gold_raw or []) if str(x).strip() }
        if not gold:
            continue

        # Retrieve candidates
        cand = retrieve_context(q, top_k=k*8, domain=domain, as_strings=False)  # overfetch, then dedup

        # Build ranked list with a stable, unique match key per item
        ranked = []
        seen = set()
        for c in cand:
            key = c.get("uri") or c.get("URL") or c.get("Url") or c.get("link") or c.get("title") or c.get("Title") or ""
            key = _norm(key)
            if not key:
                # fallback to normalized title only if nothing else
                key = _norm(c.get("title") or c.get("Title") or "")
            if not key or key in seen:
                continue
            seen.add(key)
            ranked.append(key)
            if len(ranked) >= k:  # keep only top-k after dedup
                break

        # Binary relevance for the deduped ranked list
        rel = [1 if r in gold else 0 for r in ranked]

        # Recall@k
        hit = 1 if any(rel) else 0
        Rk += hit

        # MRR@k
        rr = 0.0
        for i, flag in enumerate(rel, start=1):
            if flag:
                rr = 1.0 / i
                break
        MRR += rr

        # nDCG@k with proper normalization (binary gains)
        DCG = sum(flag / math.log2(i+1) for i, flag in enumerate(rel, start=1))
        ideal_gains = [1]*min(len(gold), k)  # how many *distinct* relevant items could appear
        IDCG = sum(g / math.log2(i+1) for i, g in enumerate(ideal_gains, start=1)) or 1.0
        nDCG = DCG / IDCG
        nDCG_sum += nDCG

        rows.append({"query": q, "hits": rel, "RR": rr, "has_hit": hit, "nDCG": round(nDCG, 6)})
        n += 1

    summary = {
        "Queries": n,
        f"Recall@{k}": round(Rk / max(n,1), 3),
        f"MRR@{k}": round(MRR / max(n,1), 3),
        f"nDCG@{k}": round(nDCG_sum / max(n,1), 3),
    }
    return pd.DataFrame(rows), summary


In [ ]:
# Pick k
K = 5

# Run eval
rows, summary = retrieval_eval(eval_df, k=K, use_uris=False, domain=None)

# --- Print summary nicely ---
print("=== Retrieval metrics ===")
for key, val in summary.items():
    print(f"{key}: {val}")

# --- Show a few per-query rows ---
from IPython.display import display
print("\n=== First 10 queries ===")
display(rows.head(10))

# --- (Optional) show queries with no hit ---
nohit = rows[rows["has_hit"] == 0]
print(f"\nQueries with NO hit in top-{K}: {len(nohit)}")
if not nohit.empty:
    display(nohit[["query", "hits", "RR", "nDCG"]])

# --- (Optional) save to files for report ---
rows.to_csv("retrieval_eval_rows.csv", index=False)
with open("retrieval_eval_summary.txt", "w") as f:
    for key, val in summary.items():
        f.write(f"{key}: {val}\n")
print("\nSaved: retrieval_eval_rows.csv and retrieval_eval_summary.txt")


In [ ]:
print(retrieval_eval(eval_df, k=6, use_uris=False, domain=None)[1])


#### Export results for appendix

In [ ]:
r_rows.to_csv("eval_retrieval_rows.csv", index=False)
g_rows.to_csv("eval_generation_rows.csv", index=False)
pd.DataFrame([r_summary]).to_csv("eval_retrieval_summary.csv", index=False)
pd.DataFrame([g_summary]).to_csv("eval_generation_summary.csv", index=False)
print("Saved CSVs for the report’s appendix.")

In [ ]:
import math, re, unicodedata, pandas as pd
from ast import literal_eval

# --- Robust normaliser: strip accents, unify quotes, kill punctuation/extra spaces, lower-case
def _norm(s):
    if s is None: return ""
    s = str(s)
    # Unicode normalise + strip accents
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    # unify curly quotes
    s = s.replace("’","'").replace("‘","'").replace("“",'"').replace("”",'"')
    s = s.lower()
    # keep letters/digits as tokens
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# --- Make sure a column value becomes a Python list of strings
def _to_list(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, (list, tuple, set)):
        return list(x)
    s = str(x).strip()
    # If it looks like a list literal, try to parse it
    if len(s) >= 2 and s[0] == "[" and s[-1] == "]":
        try:
            parsed = literal_eval(s)
            # guard: ensure list of strings
            if isinstance(parsed, (list, tuple, set)):
                return [str(y) for y in parsed]
        except Exception:
            pass
    # otherwise treat as single-item list
    return [s]

def retrieval_eval(eval_df: pd.DataFrame, k=6, use_uris=False, domain=None, debug_n=3):
    rows = []
    Rk = MRR = nDCG_sum = 0.0
    n = 0

    # small debug collector
    dbg = []

    for rec in eval_df.itertuples(index=False):
        q = getattr(rec, "query", None)
        if not q:
            continue

        # Pull gold field (prefer titles unless explicitly set use_uris=True)
        gold_col = "relevant_uris" if use_uris else "relevant_titles"
        gold = getattr(rec, gold_col, None)
        if gold is None:
            # fallback to the other column name if present
            gold = getattr(rec, "relevant_titles", None) or getattr(rec, "relevant_uris", None)

        gold_list = _to_list(gold)
        gold_norm = { _norm(g) for g in gold_list if str(g).strip() }

        if not gold_norm:
            # nothing to evaluate for this query
            continue

        # Retrieve model candidates (expects existing retrieve_context)
        cand = retrieve_context(q, top_k=k, domain=domain, as_strings=False)

        # Get comparable keys
        got_keys = []
        for c in cand:
            if use_uris:
                key_raw = c.get("uri") or c.get("URL") or ""
            else:
                key_raw = c.get("title") or c.get("Title") or ""
                if not key_raw:  # last-resort fallback
                    key_raw = c.get("uri") or c.get("URL") or ""
            got_keys.append(_norm(key_raw))

        rel = [1 if g in gold_norm else 0 for g in got_keys]

        # Recall@k
        hit = 1 if any(rel) else 0
        Rk += hit

        # MRR@k
        rr = 0.0
        for i, flag in enumerate(rel, start=1):
            if flag:
                rr = 1.0 / i
                break
        MRR += rr

        # nDCG@k (binary gains), properly normalised
        DCG = sum(flag / math.log2(i+1) for i, flag in enumerate(rel, start=1))
        IDCG = sum(1.0 / math.log2(i+1) for i in range(1, min(len(gold_norm), k)+1)) or 1.0
        nDCG = DCG / IDCG
        nDCG_sum += nDCG

        # record per-query
        rows.append({"query": q, "hits": rel, "RR": rr, "has_hit": hit, "nDCG": nDCG})
        n += 1

        # light debug for first few
        if len(dbg) < debug_n:
            dbg.append({
                "query": q,
                "gold_raw": gold_list,
                "gold_norm": sorted(gold_norm),
                "got_keys": got_keys,
                "rel": rel
            })

    # Print tiny debug to help you see mismatches
    if dbg:
        print("\n--- DEBUG (first few queries) ---")
        for d in dbg:
            print(f"\nQuery: {d['query']}")
            print(f"Gold (raw): {d['gold_raw']}")
            print(f"Gold (norm): {d['gold_norm']}")
            print(f"Got keys:   {d['got_keys']}")
            print(f"Rel flags:  {d['rel']}")

    summary = {
        "Queries": n,
        f"Recall@{k}": round(Rk / max(n,1), 4),
        f"MRR@{k}": round(MRR / max(n,1), 4),
        f"nDCG@{k}": round(nDCG_sum / max(n,1), 4),
    }
    return pd.DataFrame(rows), summary

# Evaluate at k=6 (or k=5 if preferred)
rows, summary = retrieval_eval(eval_df, k=6, use_uris=False, domain=None)
print(summary)
rows.head(10)




In [ ]:
# === Common setup ===
import os, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

FIGDIR = Path("figures"); FIGDIR.mkdir(exist_ok=True, parents=True)

# Load main corpus (adjust filename if needed)
corpus_path = "legal_corpus_domain_tagged.csv"  # or latest combined CSV
df = pd.read_csv(corpus_path)

# Normalise some expected columns
for col in ["Source","Dataset","Type","LegalDomain","Title","Text"]:
    if col not in df.columns:
        df[col] = np.nan

def savefig(name):
    plt.tight_layout()
    out = FIGDIR / name
    plt.savefig(out, dpi=200, bbox_inches="tight")
    print(f"Saved: {out}")
    plt.close()


In [ ]:
dom = df["LegalDomain"].fillna("Unknown").value_counts().sort_values()
fig, ax = plt.subplots(figsize=(8,5))
ax.barh(dom.index, dom.values)
ax.set_title("Corpus by Legal Domain")
ax.set_xlabel("Documents")
ax.grid(axis="x", alpha=0.3)

# annotate bars
for y, v in enumerate(dom.values):
    ax.annotate(str(int(v)), (v, y), va="center", ha="left", xytext=(3,0), textcoords="offset points")
plt.tight_layout()
plt.show()


fig 3

In [ ]:
mix = df["Type"].fillna("Unknown").value_counts()
fig, ax = plt.subplots(figsize=(6,6))
wedges, _ = ax.pie(mix.values, startangle=90)
# donut hole
circle = plt.Circle((0,0), 0.65, fc="white")
fig.gca().add_artist(circle)
ax.set_title("Document Type Mix")
ax.legend(wedges, mix.index, title="Type", loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()


In [ ]:
# Install once per runtime (harmless to re-run)
%pip -q install plotly umap-learn wordcloud kaleido

import os, json, math, pathlib, textwrap
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from collections import Counter, defaultdict
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Paths used across the notebook
DATA_DIR = Path("data")
CORPUS_TAGGED = Path("legal_corpus_domain_tagged.csv")
MERGED = Path("merged_legal_corpus.csv")
EMB_PATH = DATA_DIR / "legal_embeddings.npy"          # if saved embeddings earlier
META_PATH = DATA_DIR / "legal_chunks.parquet"         # if saved chunk metadata earlier
RETR_ROWS = Path("retrieval_eval_rows.csv")           # if saved the per-query rows
RETR_SUMMARY_TXT = Path("retrieval_eval_summary.txt") # optional, if saved summary

print("Setup done.")


1) Load corpus (robust to missing columns)

In [ ]:
def _load_corpus():
    if CORPUS_TAGGED.exists():
        df = pd.read_csv(CORPUS_TAGGED)
    elif MERGED.exists():
        df = pd.read_csv(MERGED)
    else:
        raise FileNotFoundError("Could not find legal_corpus_domain_tagged.csv or merged_legal_corpus.csv")

    # Normalise common columns
    cols = {c.lower(): c for c in df.columns}
    # Ensure we have Title, Text
    for must in ["title", "text"]:
        if must not in cols:
            raise ValueError(f"Corpus is missing required column: {must}")

    # Best-effort: Source / Dataset / LegalDomain
    if "source" not in cols:
        df["Source"] = df.get("Source", "Unknown")
    if "dataset" not in cols:
        df["Dataset"] = df.get("Dataset", "Unknown")
    if "legaldomain" not in cols and "LegalDomain" not in df.columns:
        df["LegalDomain"] = "Unknown"

    # Char/word length for some plots
    df["CharLen"] = df["Text"].astype(str).str.len()
    df["WordLen"] = df["Text"].astype(str).str.split().apply(len)

    return df

try:
    corpus_df = _load_corpus()
    display(corpus_df.head())
    print("Corpus loaded:", corpus_df.shape)
except Exception as e:
    print("Load error:", e)


In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# Collapse tiny sources to "Other" to keep the viz readable
min_count = 10  # tweak for corpus size
src_counts = cdf["Source"].value_counts()
small_src = src_counts[src_counts < min_count].index
cdf_plot = cdf.copy()
cdf_plot.loc[cdf_plot["Source"].isin(small_src), "Source"] = "Other sources"

flows = cdf_plot.groupby(["LegalDomain","Source"]).size().reset_index(name="count")

domains = flows["LegalDomain"].unique().tolist()
sources = flows["Source"].unique().tolist()

labels = domains + sources
idx = {name: i for i, name in enumerate(labels)}

sankey = go.Figure(data=[go.Sankey(
    node=dict(label=labels, pad=18, thickness=16),
    link=dict(
        source=[idx[d] for d in flows["LegalDomain"]],
        target=[idx[s] for s in flows["Source"]],
        value=flows["count"].tolist()
    )
)])
sankey.update_layout(title_text="Corpus flow: Domain → Source", font_size=12, height=600)
sankey.show()


3) “Fun” word clouds by domain (top 5 domains)

In [ ]:
top_domains = (
    corpus_df["LegalDomain"].fillna("Unknown").value_counts().head(5).index.tolist()
)

fig, axs = plt.subplots(1, len(top_domains), figsize=(6*len(top_domains), 6))
if len(top_domains) == 1:
    axs = [axs]

for ax, dom in zip(axs, top_domains):
    text = " ".join(corpus_df.loc[corpus_df["LegalDomain"]==dom, "Text"].astype(str).tolist())
    wc = WordCloud(width=800, height=600, background_color="white").generate(text)
    ax.imshow(wc)
    ax.axis("off")
    ax.set_title(dom)
plt.suptitle("Word clouds by legal domain", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

# 1) Load CSVs (fix any typos in paths if needed)
statutes_df     = pd.read_csv("uk_statutes_extracted.csv")          # legislation.gov.uk
hybrid_cases_df = pd.read_csv("uk_case_law_corpus_hybrid.csv")      # LawTeacher/eLawResources
bailii_df       = pd.read_csv("BAILII_Case_Law_Formatted.csv")      # BAILII
NGO_df          = pd.read_csv("gov_ngo_advice.csv")                 # GOV.UK/Citizens Advice
uksc_df         = pd.read_csv("supreme_court_cases.csv")            # UK Supreme Court

# 2) Standardise into a common schema: Title, Text, Source, Dataset (+ optional LegalDomain)
def standardize_columns(df, dataset_name, source_name, data_type=None):
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]

    # Try to find a text column
    candidates = [c for c in out.columns if c.lower() in ["text", "body", "content"]]
    if "Text" not in out.columns and candidates:
        out["Text"] = out[candidates[0]]
    if "Title" not in out.columns:
        # fallbacks for title-like columns
        title_cands = [c for c in out.columns if c.lower() in ["title","case","case_name","name","heading"]]
        if title_cands:
            out["Title"] = out[title_cands[0]]
        else:
            out["Title"] = "(untitled)"

    out["Source"]  = source_name
    out["Dataset"] = dataset_name
    if data_type is not None:
        out["Type"] = data_type

    keep = ["Title","Text","Source","Dataset","Type","LegalDomain"]
    keep = [c for c in keep if c in out.columns]
    return out[keep]

statutes_df     = standardize_columns(statutes_df,     "legislation.gov.uk",      "legislation.gov.uk",      "Statute")
hybrid_cases_df = standardize_columns(hybrid_cases_df, "LawTeacher/eLawResources","LawTeacher/eLawResources","Case Law")
bailii_df       = standardize_columns(bailii_df,       "BAILII",                  "BAILII",                  "Case Law")
NGO_df          = standardize_columns(NGO_df,          "GOV.UK",                  "GOV.UK",                  "Guidance")
uksc_df         = standardize_columns(uksc_df,         "UK Supreme Court",        "UK Supreme Court",        "Case Law")

# ✅ 3) Concatenate **all** datasets
combined_df = pd.concat(
    [statutes_df, hybrid_cases_df, bailii_df, NGO_df, uksc_df],
    ignore_index=True
)

# Basic clean
combined_df["Title"] = combined_df["Title"].astype(str).str.strip()
combined_df["Text"]  = combined_df["Text"].astype(str).str.strip()
combined_df = combined_df.dropna(subset=["Title","Text"])
print("Combined rows:", len(combined_df))
combined_df.head(3)


In [ ]:
import numpy as np

corpus_df = combined_df.copy()

# Normalise strings and reveal what’s missing
for col in ["Dataset","LegalDomain","Source"]:
    if col not in corpus_df.columns:
        corpus_df[col] = ""
    corpus_df[col] = corpus_df[col].astype(str).str.strip().replace({"nan": ""})

print("Value counts — Dataset:")
print(corpus_df["Dataset"].value_counts(dropna=False), "\n")

print("Value counts — Source:")
print(corpus_df["Source"].value_counts(dropna=False), "\n")

print("Value counts — LegalDomain (empty means missing):")
print(corpus_df["LegalDomain"].replace({"": np.nan}).value_counts(dropna=False), "\n")

missing_any = (
    (corpus_df["Dataset"]=="") |
    (corpus_df["LegalDomain"]=="") |
    (corpus_df["Source"]=="")
)
print("Rows that would be dropped by sunburst due to blanks in path:", int(missing_any.sum()))


In [ ]:
import numpy as np
import plotly.express as px

cdf = corpus_df.copy()

# Canonicalise labels so categories aren’t split
alias = {
    "BAILII.org": "BAILII",
    "bailii": "BAILII",
    "Legislation.gov.uk": "legislation.gov.uk",
    "legislation": "legislation.gov.uk",
    "Gov.uk": "GOV.UK",
    "gov.uk": "GOV.UK",
    "Supreme Court": "UK Supreme Court",
    "Hybrid": "LawTeacher/eLawResources",
}
for col in ["Dataset","Source"]:
    cdf[col] = cdf[col].replace(alias)

# Don’t let missing domains vanish—bucket them
cdf["LegalDomain"] = cdf["LegalDomain"].replace("", np.nan).fillna("Unknown")

# Filter out any rows that still lack path levels
cdf_plot = cdf[
    (cdf["Dataset"]!="") & (cdf["LegalDomain"]!="") & (cdf["Source"]!="")
].copy()

# Use explicit counts to avoid surprises
cdf_plot["count"] = 1

sun = px.sunburst(
    cdf_plot,
    path=["Dataset","LegalDomain","Source"],
    values="count",
    color="LegalDomain",
    title="Corpus composition: Dataset → LegalDomain → Source (counts)",
    branchvalues="total",
)
sun.update_layout(height=700)
sun.show()


In [ ]:
import plotly.graph_objects as go

flows = cdf_plot.groupby(["LegalDomain","Source"])["count"].sum().reset_index()
labels = pd.unique(flows[["LegalDomain","Source"]].values.ravel("K")).tolist()
idx = {s:i for i,s in enumerate(labels)}

fig = go.Figure(data=[go.Sankey(
    node=dict(label=labels, pad=18, thickness=16),
    link=dict(
        source=[idx[d] for d in flows["LegalDomain"]],
        target=[idx[s] for s in flows["Source"]],
        value=flows["count"].tolist()
    )
)])
fig.update_layout(title_text="Corpus flow: LegalDomain → Source", font_size=12, height=600)
fig.show()


In [ ]:
# === Vis stack: load, clean, combine, and plot ===
# Drop this in ONE cell. Re-run after CSVs are present in the working dir.

# Optional: quick installs (safe to leave on)
try:
    import plotly.express as px, plotly.graph_objects as go
    from wordcloud import WordCloud
    import umap
except Exception:
    %pip -q install plotly wordcloud umap-learn

import re, os, math, json, textwrap, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from wordcloud import WordCloud
try:
    import umap
except Exception:
    umap = None

# ----------------------------
# 1) Load everything robustly
# ----------------------------
def load_csv_safe(path):
    if os.path.exists(path):
        try:
            return pd.read_csv(path, dtype=str, encoding_errors="replace", low_memory=False)
        except Exception as e:
            print(f"[WARN] Could not read {path}: {e}")
    else:
        print(f"[MISS] {path} not found.")
    return pd.DataFrame()

statutes_df = load_csv_safe("uk_statutes_extracted.csv")
hybrid_cases_df = load_csv_safe("uk_case_law_corpus_hybrid.csv")
bailii_df = load_csv_safe("BAILII_Case_Law_Formatted.csv")
NGO_df = load_csv_safe("gov_ngo_advice.csv")
uksc_df = load_csv_safe("supreme_court_cases.csv")

# ----------------------------
# 2) Standardise columns
# ----------------------------
def standardize_columns(df, dataset_name, type_hint):
    if df.empty:
        return df
    # normalise column names
    df = df.rename(columns=lambda x: str(x).strip())
    # best-effort remap common variants
    colmap = {
        "title": "Title", "TITLE": "Title",
        "text": "Text", "TEXT": "Text",
        "source": "Source", "SOURCE": "Source",
        "url": "URI", "Url": "URI", "URI": "URI",
        "Type": "Type"
    }
    for c in list(df.columns):
        if c in colmap:
            df = df.rename(columns={c: colmap[c]})

    # force required cols
    for need in ["Title","Text"]:
        if need not in df.columns:
            df[need] = ""

    # carry optional URI if present
    if "URI" not in df.columns:
        df["URI"] = ""

    # Define Dataset (origin) and Source (kind)
    df["Dataset"] = dataset_name            # e.g., "Legislation.gov.uk"
    df["Source"]  = type_hint               # e.g., "Statute"/"Case Law"/"Guidance"

    # Keep just the useful columns in a stable order
    keep = ["Title","Text","Dataset","Source","URI"]
    extras = [c for c in df.columns if c not in keep]
    return df[keep + extras]

# Map each file into (Dataset, Source-kind)
statutes_df     = standardize_columns(statutes_df, "Legislation.gov.uk", "Statute")
hybrid_cases_df = standardize_columns(hybrid_cases_df, "Hybrid",             "Case Law")
bailii_df       = standardize_columns(bailii_df, "BAILII.org",               "Case Law")
NGO_df          = standardize_columns(NGO_df, "Gov/CitizensAdvice",          "Guidance")
uksc_df         = standardize_columns(uksc_df, "UK Supreme Court",           "Case Law")

# ----------------------------
# 3) Combine exactly as asked
# ----------------------------
combined_df = pd.concat(
    [statutes_df, hybrid_cases_df, bailii_df, NGO_df, uksc_df],
    ignore_index=True
)

# ----------------------------
# 4) Clean + light normalisation
# ----------------------------
def _bad(s):
    s = str(s).strip().lower()
    return (s in {"", "no title found", "no text extracted", "nan", "none", "null"}
            or "access denied" in s)

mask = (~combined_df["Title"].map(_bad)) & (~combined_df["Text"].map(_bad))
combined_df = combined_df.loc[mask].copy()

# Remove obvious navigation boilerplate (BAILII headers etc.)
combined_df["Text"] = combined_df["Text"].astype(str).str.replace(
    r"\[\s*Home\s*\].*?\[\s*Databases\s*\].*?\[\s*World Law\s*\].*?\n", "",
    regex=True
)

# ----------------------------
# 5) Add LegalDomain if missing
# ----------------------------
if "LegalDomain" not in combined_df.columns:
    keyword_scores = {
        "Contract": ["contract","breach","offer","acceptance","consideration","terms",
                     "frustration","misrepresentation","invitation to treat","estoppel","agreement"],
        "Tort": ["negligence","duty of care","liability","causation","damage","remoteness",
                 "nuisance","trespass","occupier","personal injury","vicarious","defamation"],
        "Employment": ["employee","employer","dismissal","redundancy","tribunal",
                       "discrimination","wages","employment rights"],
        "Consumer": ["consumer","goods","trader","unfair","product","merchantable"],
        "Property": ["lease","tenancy","land","mortgage","easement","freehold","title"],
    }
    def tag_domain(row):
        txt = f"{row['Title']} {row['Text']}".lower()
        scores = {k:0 for k in keyword_scores}
        for dom, kws in keyword_scores.items():
            for kw in kws:
                # cheap but effective
                if re.search(rf"\b{re.escape(kw)}\b", txt):
                    scores[dom] += 1
        m = max(scores.values()) if scores else 0
        if m == 0: return "Unknown"
        tops = [d for d,v in scores.items() if v == m]
        return tops[0] if len(tops) == 1 else "Mixed"

    combined_df["LegalDomain"] = combined_df.apply(tag_domain, axis=1)

# For compact legends
combined_df["LegalDomain"] = combined_df["LegalDomain"].fillna("Unknown")
combined_df["Dataset"]     = combined_df["Dataset"].fillna("Unknown")
combined_df["Source"]      = combined_df["Source"].fillna("Unknown")

# Also keep a working alias many of the viz snippets used
corpus_df = combined_df.copy()

print(f"Loaded rows: {len(corpus_df):,}")
print(corpus_df[["Dataset","Source","LegalDomain"]].head())

# ===========================
# BEAUTIFUL VISUALISATIONS
# ===========================

# ---- V1. Sunburst: Dataset → LegalDomain → Source ----
if {"Dataset","LegalDomain","Source"}.issubset(corpus_df.columns):
    sun = px.sunburst(
        corpus_df,
        path=["Dataset","LegalDomain","Source"],
        title="Corpus composition: Dataset → LegalDomain → Source",
        color="LegalDomain",
        branchvalues="total",
        height=600
    )
    sun.update_layout(margin=dict(t=80,l=0,r=0,b=0))
    sun.show()
else:
    print("[SKIP] Sunburst: requires Dataset, LegalDomain, Source columns")

# ---- V2. Sankey: LegalDomain → Dataset (group smalls as 'Other') ----
if {"LegalDomain","Dataset"}.issubset(corpus_df.columns):
    flows = (corpus_df
             .groupby(["LegalDomain","Dataset"]).size()
             .reset_index(name="count")
             .sort_values("count", ascending=False))

    # keep top 10 datasets, bucket the rest
    top_ds = flows["Dataset"].value_counts().index[:10].tolist()
    flows["DatasetG"] = flows["Dataset"].where(flows["Dataset"].isin(top_ds), "Other")

    flows_g = (flows.groupby(["LegalDomain","DatasetG"])["count"].sum()
                    .reset_index())

    domains = flows_g["LegalDomain"].unique().tolist()
    dsets   = flows_g["DatasetG"].unique().tolist()
    labels  = domains + dsets
    idx = {name:i for i,name in enumerate(labels)}

    sankey = go.Figure(data=[go.Sankey(
        node=dict(label=labels, pad=20, thickness=18),
        link=dict(
            source=[idx[d] for d in flows_g["LegalDomain"]],
            target=[idx[s] for s in flows_g["DatasetG"]],
            value=flows_g["count"].tolist()
        )
    )])
    sankey.update_layout(title_text="Flow of content: LegalDomain → Dataset", font_size=12, height=600)
    sankey.show()
else:
    print("[SKIP] Sankey: requires LegalDomain and Dataset columns")

# ---- V3. Treemap: Source (kind) → Dataset with text size ----
if {"Source","Dataset","Text"}.issubset(corpus_df.columns):
    tmp = corpus_df.copy()
    tmp["approx_words"] = tmp["Text"].astype(str).str.split().map(len)
    tree = px.treemap(
        tmp, path=["Source","Dataset"], values="approx_words",
        title="Treemap: Source → Dataset (area ≈ total words)",
        height=620
    )
    tree.update_traces(textinfo="label+value")
    tree.show()
else:
    print("[SKIP] Treemap: requires Source, Dataset, Text columns")

# ---- V4. Timeline (rough): year extracted from Title ----
def extract_year(s):
    m = re.search(r"(18|19|20)\d{2}", str(s))
    return int(m.group(0)) if m else np.nan

timeline = corpus_df.copy()
timeline["Year"] = corpus_df["Title"].map(extract_year)
year_ct = (timeline.dropna(subset=["Year"])
                    .groupby(["Year","LegalDomain"]).size()
                    .reset_index(name="count"))
if not year_ct.empty:
    line = px.line(year_ct, x="Year", y="count", color="LegalDomain",
                   markers=True, title="Items over time by LegalDomain")
    line.update_layout(height=480)
    line.show()
else:
    print("[SKIP] Timeline: no parseable years in Title.")

# ---- V5. Heatmap: Domain × Dataset (counts) ----
if {"LegalDomain","Dataset"}.issubset(corpus_df.columns):
    ct = pd.crosstab(corpus_df["LegalDomain"], corpus_df["Dataset"])
    heat = px.imshow(ct, text_auto=True, aspect="auto",
                     title="Heatmap: LegalDomain × Dataset (counts)")
    heat.update_layout(height=600)
    heat.show()
else:
    print("[SKIP] Heatmap: requires LegalDomain and Dataset columns")

# ---- V6. WordClouds per LegalDomain (top 5 domains) ----
try:
    import matplotlib.pyplot as plt
    from collections import Counter
    top_domains = (corpus_df["LegalDomain"].value_counts().index[:5]).tolist()
    fig_ct = min(5, len(top_domains))
    if fig_ct:
        fig, axes = plt.subplots(1, fig_ct, figsize=(4*fig_ct, 3), tight_layout=True)
        if fig_ct == 1: axes = [axes]
        for ax, dom in zip(axes, top_domains):
            text = " ".join(corpus_df.loc[corpus_df["LegalDomain"]==dom, "Text"].astype(str).tolist())
            wc = WordCloud(width=600, height=400, background_color="white", collocations=False).generate(text)
            ax.imshow(wc); ax.axis("off"); ax.set_title(dom)
        plt.suptitle("Word clouds by LegalDomain", y=1.02)
        plt.show()
    else:
        print("[SKIP] WordCloud: no domains.")
except Exception as e:
    print(f"[SKIP] WordClouds (matplotlib/wordcloud issue): {e}")

# ---- V7. UMAP scatter of text embeddings (sampled) ----
# Uses data/legal_embeddings.npy & data/legal_chunks.parquet if present,
# else computes a quick sample on the fly.
def try_umap_plot():
    if umap is None:
        print("[SKIP] UMAP: umap-learn not available")
        return
    # Preferred: use precomputed files
    emb_path = "data/legal_embeddings.npy"
    meta_path = "data/legal_chunks.parquet"
    if os.path.exists(emb_path) and os.path.exists(meta_path):
        embs = np.load(emb_path)
        meta = pd.read_parquet(meta_path)
        m = min(len(meta), len(embs))
        embs, meta = embs[:m], meta.iloc[:m]
        labels = (meta["domain"] if "domain" in meta.columns else
                  meta["LegalDomain"] if "LegalDomain" in meta.columns else
                  pd.Series(["Unknown"]*m))
    else:
        # fallback: compute on a small sample using MiniLM
        try:
            from sentence_transformers import SentenceTransformer
        except Exception:
            %pip -q install sentence-transformers
            from sentence_transformers import SentenceTransformer

        sample = corpus_df.sample(min(1000, len(corpus_df)), random_state=42)
        texts  = sample["Text"].astype(str).str.slice(0, 2000).tolist()
        model  = SentenceTransformer("all-MiniLM-L6-v2")
        embs   = model.encode(texts, normalize_embeddings=True, convert_to_numpy=True)
        labels = sample["LegalDomain"].reset_index(drop=True)

    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=20, min_dist=0.2)
    coords  = reducer.fit_transform(embs)
    df2d    = pd.DataFrame({"x": coords[:,0], "y": coords[:,1], "label": labels})

    sc = px.scatter(df2d, x="x", y="y", color="label",
                    title="UMAP of embeddings (colored by LegalDomain)",
                    opacity=0.85, height=600)
    sc.update_traces(marker=dict(size=6))
    sc.show()

try_umap_plot()

# ---- V8. “Interestingness” bar: longest titles per Dataset ----
tmp = corpus_df.copy()
tmp["TitleLen"] = tmp["Title"].astype(str).str.len()
top_long = (tmp.sort_values("TitleLen", ascending=False)
              .groupby("Dataset").head(5)
              .sort_values(["Dataset","TitleLen"], ascending=[True,False]))
bar = px.bar(top_long, x="TitleLen", y="Title", color="Dataset",
             orientation="h", title="Fun: Longest titles by dataset (top-5 each)", height=800)
bar.update_layout(yaxis={'categoryorder':'total ascending'})
bar.show()

print("\n[OK] Visualisations complete.")


In [ ]:
# Timeline (rough): year extracted from Title
def extract_year(s):
    m = re.search(r"(18|19|20)\d{2}", str(s))
    return int(m.group(0)) if m else np.nan

timeline = corpus_df.copy()
timeline["Year"] = corpus_df["Title"].map(extract_year)

year_ct = (timeline.dropna(subset=["Year"])
                    .groupby(["Year","LegalDomain"]).size()
                    .reset_index(name="count"))

if not year_ct.empty:
    # Okabe–Ito palette (colorblind friendly)
    palette = ['#0072B2', '#D55E00', '#009E73', '#CC79A7',
               '#F0E442', '#56B4E9', '#E69F00', '#000000']

    # Deterministic mapping from domains to colors
    domains = sorted(year_ct["LegalDomain"].dropna().unique())
    color_map = {d: palette[i % len(palette)] for i, d in enumerate(domains)}

    line = px.line(
        year_ct, x="Year", y="count", color="LegalDomain",
        color_discrete_map=color_map, markers=True,
        title="Items over time by LegalDomain"
    )
    line.update_layout(height=480)
    line.show()
else:
    print("[SKIP] Timeline: no parseable years in Title.")


In [ ]:
def retrieval_eval(eval_df: pd.DataFrame, k=6, use_uris=False, domain=None):
    rows = []
    Rk = MRR = nDCG_sum = 0.0
    n = 0

    def _dcg(rels):
        # rels is a list of 0/1 of length k
        return sum(v / math.log2(i + 1) for i, v in enumerate(rels, start=1))

    for rec in eval_df.itertuples():
        q = rec.query
        gold = getattr(rec, "relevant_uris", None) if use_uris else getattr(rec, "relevant_titles", None)
        if gold is None:  # fallback to whichever exists
            gold = getattr(rec, "relevant_titles", None) or getattr(rec, "relevant_uris", None)

        gold = set(_norm(x) for x in (gold or []) if str(x).strip())
        if not gold:
            continue

        # Retrieve
        cand = retrieve_context(q, top_k=k, domain=domain, as_strings=False)

        # Keys to match against gold (title -> uri fallback)
        got_keys = []
        for c in cand:
            key = _norm(c.get("uri") or "") if use_uris else _norm(c.get("title") or c.get("Title") or "")
            if not key:
                key = _norm(c.get("title") or c.get("Title") or c.get("uri") or "")
            got_keys.append(key)

        # Build binary relevance with de-dup + pad to k
        seen = set()
        rel = []
        for key in got_keys:
            if key in seen:
                rel.append(0)          # don’t double-count the same item
            else:
                seen.add(key)
                rel.append(1 if key in gold else 0)
            if len(rel) == k:
                break
        if len(rel) < k:
            rel += [0] * (k - len(rel))

        # Hit@k (kept label "Recall@k" to match original summary)
        hit = 1 if any(rel) else 0
        Rk += hit

        # MRR@k
        rr = 0.0
        for i, flag in enumerate(rel, start=1):
            if flag:
                rr = 1.0 / i
                break
        MRR += rr

        # nDCG@k (binary gains) — fixed IDCG and stable denominator
        DCG = _dcg(rel)
        ideal_rel = [1] * min(k, len(gold)) + [0] * (k - min(k, len(gold)))
        IDCG = _dcg(ideal_rel) or 1.0
        ndcg = DCG / IDCG
        nDCG_sum += ndcg

        rows.append({"query": q, "hits": rel, "RR": rr, "has_hit": hit, f"nDCG@{k}": ndcg})
        n += 1

    summary = {
        "Queries": n,
        f"Recall@{k}": round(Rk / max(n, 1), 4),  # a.k.a. Hit@k here
        f"MRR@{k}": round(MRR / max(n, 1), 4),
        f"nDCG@{k}": round(nDCG_sum / max(n, 1), 4),
    }
    return pd.DataFrame(rows), summary

# Run it
r_rows, r_summary = retrieval_eval(eval_df, k=6, use_uris=False, domain=None)
print(r_summary)
r_rows.head()
